# 43008 Reinforcement Learning -- Assignment 3, Part E: Value-Based Deep Reinforcement Learning on Snake

**The Deadly Triad (project 48): Yue Liu (25211451), Jiahao Liu (14191136), Siqin Jin (26145324)**

**Project**: a controlled study of state representation, reward design and DQN stabilisation on a self-built Snake
environment (8 x 8 and 10 x 10 boards): tabular Q-learning; DQN on an 11-feature vector (observation A, MLP) or an
8-plane grid (observation B8, CNN); the discount factor; the replay buffer and the target network; Double DQN, Dueling
and prioritised replay; four reward designs (R0 sparse, R1 step penalty, R2 potential-based shaping, R3 naive distance
bonus) and a reward-hacking probe; random, BFS-greedy and Hamiltonian-cycle reference players. Every configuration is
trained with five seeds, the seed is the unit of replication and every interval is a 95 % Student-t interval over seeds.

**The code**: Sections 2 to 4 are the code that trained the study. Each code cell is one module of the `snake_rl`
package (repository: https://github.com/Cureeeeeeee/Value-Based-Deep-Reinforcement-Learning-on-Snake), inlined
character for character apart from the module docstring, the import statements (gathered in Section 1) and the
command-line wrappers: the environment and its two observation encodings and four reward variants, the reference
players, tabular Q-learning, the replay buffers, the Q-networks, the DQN agent, the experiment configuration, the
evaluation protocol and the training loop. `scripts/build_notebook.py` writes this notebook from those modules and
`tests/test_m10_notebook.py` fails if the two ever differ. The analysis modules (`snake_rl.analysis`: aggregation
over seeds, figures, behaviour analysis, the experiment matrix, `snake_rl.analysis.report` and
`snake_rl.analysis.report_tables`), `snake_rl.utils` and the pygame GUI (`snake_rl.gui`) are imported from the
repository and not repeated here.

**What the notebook does**: Section 5 trains with the code above, in this session: the 8 x 8 tabular agent of the
report and one DQN run on each observation (short configurations by default, the report's 8 x 8 configurations with
`TRAIN_MODE = "full"`), evaluated against the three reference players on the same boards. Sections 6 to 10 then take
the training logs of the whole study (64 configurations, 5 seeds each) and regenerate every table and figure of the
report, show them in the report's order with the report's captions, and check every computed table against the
report source cell by cell: no number of the report is typed by hand. Section 11 runs the behaviour analysis behind
Table 9 and Figures 11 to 13 on one trained network.

**Training logs**: `runs/` is not in the repository (about 1 GB with checkpoints). Three ways to get the logs into
this session: (1) upload `runs_logs.zip`, made with `python scripts/export_logs.py --include-behaviour` on the machine
that trained, when the cell of Section 6 asks for it; (2) set `LOGS_URL` in that cell to a link to the zip (for
example a GitHub release asset); (3) run the notebook inside a checkout that already has `runs/`. The zip holds the
per-run logs, the reference-player evaluations, the saved behaviour analyses (Table 9, Figures 11 to 13) and the
screenshot of Figure 1; it carries no weights.

### 1. Repository, Imports and Settings

In [ ]:
# Clone the repository and install the package on Google Colab; a local checkout is used as it is, also when opened from notebooks/
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Cureeeeeeee/Value-Based-Deep-Reinforcement-Learning-on-Snake.git"

if os.path.isdir(os.path.join("..", "snake_rl")):
    os.chdir("..")
elif not os.path.isdir("snake_rl"):
    if not os.path.isdir("snake-rl"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "snake-rl"], check=True)
    os.chdir("snake-rl")
if IN_COLAB:
    %pip install -q -e .
print("working directory:", os.getcwd())

In [ ]:
# Every import of the modules below, gathered here, plus the display helpers and the analysis modules of the repository
from __future__ import annotations

import argparse
import copy
import csv
import glob
import json
import math
import os
import platform
import re
import shutil
import sys
import time
import urllib.request
import zipfile
from collections import deque
from collections.abc import Callable, Iterator, Mapping
from contextlib import contextmanager
from dataclasses import asdict, dataclass, field, fields, replace
from datetime import datetime
from functools import partial
from pathlib import Path
from typing import Any, Literal, Protocol, Sequence, TYPE_CHECKING, get_args, runtime_checkable

import gymnasium as gym
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from IPython.display import Image, Markdown, display
from gymnasium import spaces
from torch import nn

from snake_rl.analysis import report_tables as rt
from snake_rl.analysis.plots import plot_learning_curves
from snake_rl.utils import git_hash

REPORT_MD = "docs/PartE-final-report.md"   # the report source: captions, figure files and the tables to check against
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"python {sys.version.split()[0]}  torch {torch.__version__}  numpy {np.__version__}  device {DEVICE}")

### 2. The Snake Environment (`snake_rl/env`)

The game is written once: the rules in `common.py`, the observation encoders in `obs.py` and the reward variants in
`rewards.py` are shared by the scalar reference environment (`core.py`, a `gymnasium.Env` used by the tests, the GUI
and the evaluation) and by the vectorised training engine (`vec_env.py`), which must agree with the scalar one step by
step (`tests/test_vec_env.py` is the regression gate).

#### 2.1 Rules shared by both environments

Grid cell `(r, c)` with `r` growing downwards; headings clockwise (0 = up, 1 = right, 2 = down, 3 = left); two action
modes (relative: straight / left / right, the mode of the study; absolute: the four headings). The snake is one
`W x W` integer grid `body_age`: 0 = empty, `k > 0` = this cell stays occupied for `k` more steps, so the tail is 1,
the neck `length - 1` and the head `length`. Every helper takes arrays with a leading `N` axis; the scalar environment
calls them with `N = 1`.

In [ ]:
# Rules shared by both environments: headings, actions, food sampling and the body_age grid
from __future__ import annotations

# --- headings -----------------------------------------------------------------
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
HEADING_NAMES = ("up", "right", "down", "left")
#: (dr, dc) for each heading, indexed by heading id.
DELTAS = np.array([(-1, 0), (0, 1), (1, 0), (0, -1)], dtype=np.int64)

# --- actions ------------------------------------------------------------------
STRAIGHT, TURN_LEFT, TURN_RIGHT = 0, 1, 2
#: Heading change for each relative action: straight 0, left -1, right +1.
RELATIVE_TURN = np.array([0, -1, 1], dtype=np.int64)
ActionMode = Literal["relative", "absolute"]

# --- episode end causes ---------------------------------------------------------
CAUSE_NONE, CAUSE_WALL, CAUSE_SELF, CAUSE_STARVE, CAUSE_WIN = range(5)
CAUSE_NAMES = ("none", "wall", "self", "starve", "win")
#: Causes that set ``terminated`` (next-state value is zero). ``starve`` sets
#: ``truncated`` instead and must be bootstrapped.
TERMINAL_CAUSES = (CAUSE_WALL, CAUSE_SELF, CAUSE_WIN)

#: dtype of ``body_age`` grids.
AGE_DTYPE = np.int32


def n_actions(action_mode: ActionMode) -> int:
    if action_mode == "relative":
        return 3
    if action_mode == "absolute":
        return 4
    raise ValueError(f"unknown action_mode {action_mode!r}")


def default_starve_limit(W: int) -> int:
    """Starvation limit H = max(100, W^2).

    On 10x10 a Hamiltonian player needs at most 99 steps between foods, so
    100 never kills it; on 16x16 the cycle has 256 cells, hence W^2.
    """
    return max(100, W * W)


def next_heading(heading, action, action_mode: ActionMode):
    """New heading after ``action``. Works element-wise on arrays.

    Relative mode: 0 straight, 1 turn left, 2 turn right.
    Absolute mode: action is the desired heading; the 180-degree reversal is
    treated as "straight" instead of an instant death (PLAN.md section 3.2).
    """
    heading = np.asarray(heading, dtype=np.int64)
    action = np.asarray(action, dtype=np.int64)
    if action_mode == "relative":
        return (heading + RELATIVE_TURN[action]) % 4
    if action_mode == "absolute":
        reverse = (heading + 2) % 4
        return np.where(action == reverse, heading, action)
    raise ValueError(f"unknown action_mode {action_mode!r}")


def heading_to_action(current: int, desired: int, action_mode: ActionMode) -> int:
    """Action that turns ``current`` heading into ``desired`` (used by planners).

    Relative mode cannot express the 180-degree reversal; asking for it is a
    caller bug and raises.
    """
    if action_mode == "absolute":
        return int(desired)
    if action_mode != "relative":
        raise ValueError(f"unknown action_mode {action_mode!r}")
    diff = (int(desired) - int(current)) % 4
    if diff == 0:
        return STRAIGHT
    if diff == 3:
        return TURN_LEFT
    if diff == 1:
        return TURN_RIGHT
    raise ValueError(f"heading {desired} is the reversal of {current}; not expressible as a relative action")


def initial_cells(W: int) -> list[tuple[int, int]]:
    """Initial snake, tail -> head: length 3, horizontally centred, heading right."""
    r, c = W // 2, W // 2
    return [(r, c - 2), (r, c - 1), (r, c)]


def body_age_from_cells(cells: Sequence[tuple[int, int]], W: int) -> np.ndarray:
    """Build a ``body_age`` grid from cells listed tail -> head."""
    body_age = np.zeros((W, W), dtype=AGE_DTYPE)
    for age, (r, c) in enumerate(cells, start=1):
        if body_age[r, c] != 0:
            raise ValueError(f"cell {(r, c)} listed twice")
        body_age[r, c] = age
    return body_age


def cells_from_body_age(body_age: np.ndarray) -> list[tuple[int, int]]:
    """Snake cells ordered tail -> head (ascending age)."""
    rows, cols = np.nonzero(body_age)
    order = np.argsort(body_age[rows, cols], kind="stable")
    return [(int(r), int(c)) for r, c in zip(rows[order], cols[order])]


def infer_heading(body_age: np.ndarray) -> int:
    """Heading implied by neck -> head. Requires length >= 2."""
    cells = cells_from_body_age(body_age)
    if len(cells) < 2:
        raise ValueError("need at least two cells to infer a heading")
    (nr, nc), (hr, hc) = cells[-2], cells[-1]
    delta = (hr - nr, hc - nc)
    for h, (dr, dc) in enumerate(DELTAS.tolist()):
        if (dr, dc) == delta:
            return h
    raise ValueError(f"neck {cells[-2]} and head {cells[-1]} are not adjacent")


def head_positions(body_age: np.ndarray) -> np.ndarray:
    """``(N, 2)`` head coordinates of a batch ``(N, W, W)`` of body_age grids."""
    N, W, _ = body_age.shape
    flat = body_age.reshape(N, -1).argmax(axis=1)
    return np.stack(np.divmod(flat, W), axis=1)


def manhattan(a, b) -> np.ndarray:
    """Manhattan distance between coordinate arrays whose last axis is (r, c)."""
    a = np.asarray(a, dtype=np.int64)
    b = np.asarray(b, dtype=np.int64)
    return np.abs(a - b).sum(axis=-1)


def sample_food(rng: np.random.Generator, body_age: np.ndarray) -> tuple[int, int]:
    """Uniformly random empty cell.

    Protocol (PLAN.md section 3.6): draw one random key per cell, set occupied
    cells to +inf, take the argmin. The vectorised environment uses the same
    draw with the same shape so that the scalar/vectorised equivalence test
    holds for the same seed. The caller must guarantee at least one empty cell.
    """
    keys = rng.random(body_age.shape)
    keys[body_age > 0] = np.inf
    flat = int(np.argmin(keys))
    r, c = divmod(flat, body_age.shape[1])
    return int(r), int(c)

#### 2.2 Observations A and B

Observation **A** is the community-standard hand-crafted vector of 11 binary features: danger straight / left / right,
heading one-hot (4), food above / below / left / right of the head (4); it discards the body layout and is not Markov.
Observation **B** is a multi-plane grid `(C, W, W)`: head, body, food, `body_age / length`, and four broadcast heading
planes (8 planes; switches drop the age plane and the heading planes). The age plane is what makes the state Markov:
it recovers the neck and the tail.

In [ ]:
# Observation A (11 binary features) and observation B (multi-plane grid), batched encoders
from __future__ import annotations

N_FEATURES_A = 11
#: Feature indices of observation A (kept next to the encoder for the GUI).
A_DANGER_STRAIGHT, A_DANGER_LEFT, A_DANGER_RIGHT = 0, 1, 2
A_HEADING = slice(3, 7)  # one-hot, heading order up/right/down/left
A_FOOD_UP, A_FOOD_DOWN, A_FOOD_LEFT, A_FOOD_RIGHT = 7, 8, 9, 10


@dataclass(frozen=True)
class ObsConfig:
    kind: Literal["A", "B"] = "A"
    #: B only: include ``body_age / length`` plane (Markov fix).
    age_plane: bool = True
    #: B only: include 4 broadcast heading planes (redundant given age plane).
    heading_planes: bool = True

    @property
    def n_planes(self) -> int:
        return 3 + int(self.age_plane) + 4 * int(self.heading_planes)

    def shape(self, W: int) -> tuple[int, ...]:
        if self.kind == "A":
            return (N_FEATURES_A,)
        if self.kind == "B":
            return (self.n_planes, W, W)
        raise ValueError(f"unknown observation kind {self.kind!r}")


def cell_is_danger(body_age: np.ndarray, cells: np.ndarray) -> np.ndarray:
    """True where moving into ``cells`` ``(N, 2)`` would end the episode.

    A cell is dangerous if it is off-grid or holds body with ``age > 1``.
    The tail cell (age 1) vacates during the move (PLAN.md section 3.3, step 4)
    unless the snake eats, and food is never on the body, so age 1 is always
    safe. This matches ``step`` exactly; ``tests/test_obs.py`` checks it.
    """
    N, W, _ = body_age.shape
    r, c = cells[:, 0], cells[:, 1]
    in_bounds = (r >= 0) & (r < W) & (c >= 0) & (c < W)
    val = body_age[np.arange(N), np.clip(r, 0, W - 1), np.clip(c, 0, W - 1)]
    return ~in_bounds | (val > 1)


def encode_A(body_age: np.ndarray, heading: np.ndarray, food: np.ndarray) -> np.ndarray:
    """Observation A for a batch. ``body_age (N,W,W)``, ``heading (N,)``, ``food (N,2)``."""
    N = body_age.shape[0]
    heading = np.asarray(heading, dtype=np.int64)
    food = np.asarray(food, dtype=np.int64)
    head = head_positions(body_age)
    out = np.zeros((N, N_FEATURES_A), dtype=np.float32)
    for k, turn in enumerate((0, -1, 1)):  # straight, left, right
        h = (heading + turn) % 4
        out[:, k] = cell_is_danger(body_age, head + DELTAS[h])
    out[np.arange(N), 3 + heading] = 1.0
    out[:, A_FOOD_UP] = food[:, 0] < head[:, 0]
    out[:, A_FOOD_DOWN] = food[:, 0] > head[:, 0]
    out[:, A_FOOD_LEFT] = food[:, 1] < head[:, 1]
    out[:, A_FOOD_RIGHT] = food[:, 1] > head[:, 1]
    return out


def encode_B(
    body_age: np.ndarray,
    heading: np.ndarray,
    food: np.ndarray,
    length: np.ndarray,
    cfg: ObsConfig,
) -> np.ndarray:
    """Observation B ``(N, C, W, W)`` float32 for a batch."""
    N, W, _ = body_age.shape
    heading = np.asarray(heading, dtype=np.int64)
    food = np.asarray(food, dtype=np.int64)
    length = np.asarray(length, dtype=np.float32)
    idx = np.arange(N)
    head = head_positions(body_age)

    planes: list[np.ndarray] = []
    head_plane = np.zeros((N, 1, W, W), dtype=np.float32)
    head_plane[idx, 0, head[:, 0], head[:, 1]] = 1.0
    planes.append(head_plane)
    planes.append((body_age > 0).astype(np.float32)[:, None])
    food_plane = np.zeros((N, 1, W, W), dtype=np.float32)
    food_plane[idx, 0, food[:, 0], food[:, 1]] = 1.0
    planes.append(food_plane)
    if cfg.age_plane:
        planes.append((body_age.astype(np.float32) / length[:, None, None])[:, None])
    if cfg.heading_planes:
        one_hot = np.zeros((N, 4), dtype=np.float32)
        one_hot[idx, heading] = 1.0
        planes.append(np.broadcast_to(one_hot[:, :, None, None], (N, 4, W, W)))
    return np.concatenate(planes, axis=1)


def encode(
    cfg: ObsConfig,
    body_age: np.ndarray,
    heading: np.ndarray,
    food: np.ndarray,
    length: np.ndarray,
) -> np.ndarray:
    """Dispatch on ``cfg.kind``. All inputs batched."""
    if cfg.kind == "A":
        return encode_A(body_age, heading, food)
    if cfg.kind == "B":
        return encode_B(body_age, heading, food, length, cfg)
    raise ValueError(f"unknown observation kind {cfg.kind!r}")

#### 2.3 Reward variants R0 to R3

R0: +1 food, -1 death (wall or self), 0 otherwise; starvation truncation is not a death and gets no -1, the learner
must bootstrap it. R1: R0 minus a step penalty. R2: R0 plus potential-based shaping `gamma * Phi(s') - Phi(s)` with
`Phi = -manhattan(head, food) / (2 (W - 1))` (Ng et al. 1999: the optimal policy is unchanged). R3: R0 plus a naive
distance bonus (`closer` when the move reduced the distance to the food, else `farther`); the default asymmetric
version (+0.01 / 0) makes oscillating a net gain, which is the reward-hacking probe of the report.

In [ ]:
# Reward variants R0 to R3, computed from the transition in batched form
from __future__ import annotations

RewardVariant = Literal["R0", "R1", "R2", "R3"]


@dataclass(frozen=True)
class RewardConfig:
    variant: RewardVariant = "R0"
    food: float = 1.0
    death: float = -1.0
    #: R1 per-step penalty (subtracted).
    step_penalty: float = 0.01
    #: R2 discount; must match the agent's gamma.
    gamma: float = 0.99
    #: R3 bonus for moving closer / farther.
    closer: float = 0.01
    farther: float = 0.0


def potential(dist, W: int) -> np.ndarray:
    """Phi(s) = -d / (2 (W - 1)) in [-1, 0]; 2 (W - 1) is the largest distance."""
    return -np.asarray(dist, dtype=np.float32) / np.float32(2 * (W - 1))


def compute_reward(
    cfg: RewardConfig,
    W: int,
    *,
    ate,
    died,
    terminal,
    d_prev,
    d_moved,
    d_next,
) -> np.ndarray:
    """Reward for one transition (batched).

    Parameters
    ----------
    ate       : bool, the head reached the food.
    died      : bool, terminated by wall or self collision (gets ``death``).
    terminal  : bool, ``terminated`` for any cause (wall / self / win);
                Phi(s') = 0. Truncation is *not* terminal here.
    d_prev    : Manhattan(head, food) before the move.
    d_moved   : Manhattan(new_head, food-before-move); always d_prev +- 1.
    d_next    : Manhattan(new_head, food-after-move); differs from d_moved
                only after eating. Ignored where ``terminal``.
    """
    ate = np.asarray(ate, dtype=bool)
    died = np.asarray(died, dtype=bool)
    terminal = np.asarray(terminal, dtype=bool)
    d_prev = np.asarray(d_prev, dtype=np.int64)
    d_moved = np.asarray(d_moved, dtype=np.int64)
    d_next = np.asarray(d_next, dtype=np.int64)

    zero = np.float32(0.0)
    r = np.where(ate, np.float32(cfg.food), zero) + np.where(died, np.float32(cfg.death), zero)
    if cfg.variant == "R0":
        return r.astype(np.float32)
    if cfg.variant == "R1":
        return (r - np.float32(cfg.step_penalty)).astype(np.float32)
    if cfg.variant == "R2":
        phi_next = np.where(terminal, zero, potential(d_next, W))
        return (r + np.float32(cfg.gamma) * phi_next - potential(d_prev, W)).astype(np.float32)
    if cfg.variant == "R3":
        shaping = np.where(d_moved < d_prev, np.float32(cfg.closer), np.float32(cfg.farther))
        return (r + shaping).astype(np.float32)
    raise ValueError(f"unknown reward variant {cfg.variant!r}")

#### 2.4 The scalar reference environment

One board behind the Gymnasium API. The step order is the rule set: the new heading from the action; off-grid ends
the episode (`wall`); the food cell is eaten (the tail stays, `length += 1`, new food on a free cell; a full board
is a `win`); otherwise every positive cell is decremented so the tail vacates, a positive head cell ends the episode
(`self`), moving into the vacating tail cell is legal; `H` steps without food truncate the episode (`starve`), which
is not a terminal state because `steps_since_food` is not observable.

In [ ]:
# The scalar reference environment (gymnasium.Env): one board, the step order is the rule set
from __future__ import annotations


@dataclass(frozen=True)
class EnvConfig:
    W: int = 10
    action_mode: ActionMode = "relative"
    #: Starvation limit H; ``None`` -> ``max(100, W*W)``.
    starve_limit: int | None = None
    obs: ObsConfig = field(default_factory=ObsConfig)
    reward: RewardConfig = field(default_factory=RewardConfig)

    @property
    def H(self) -> int:
        return self.starve_limit if self.starve_limit is not None else default_starve_limit(self.W)


class SnakeEnv(gym.Env):
    """Single Snake board. See module docstring for the rules."""

    metadata = {"render_modes": ["ansi"], "render_fps": 10}

    def __init__(self, cfg: EnvConfig | None = None, render_mode: str | None = None) -> None:
        super().__init__()
        self.cfg = cfg if cfg is not None else EnvConfig()
        if self.cfg.W < 4:
            raise ValueError("W must be at least 4")
        self.W = self.cfg.W
        self.H = self.cfg.H
        self.render_mode = render_mode
        self.action_space = spaces.Discrete(n_actions(self.cfg.action_mode))
        self.observation_space = spaces.Box(0.0, 1.0, self.cfg.obs.shape(self.W), dtype=np.float32)

        self._rng: np.random.Generator | None = None
        self.body_age: np.ndarray = np.zeros((self.W, self.W), dtype=AGE_DTYPE)
        self.heading: int = RIGHT
        self.food: tuple[int, int] = (0, 0)
        self.length: int = 0
        self.score: int = 0
        self.steps_since_food: int = 0
        self.steps: int = 0
        self._done: bool = True

    # ------------------------------------------------------------------ API
    def reset(
        self, *, seed: int | None = None, options: dict[str, Any] | None = None
    ) -> tuple[np.ndarray, dict[str, Any]]:
        super().reset(seed=seed)
        if seed is not None or self._rng is None:
            self._rng = np.random.default_rng(seed)
        cells = initial_cells(self.W)
        self.body_age = body_age_from_cells(cells, self.W)
        self.heading = RIGHT
        self.length = len(cells)
        self.score = 0
        self.steps_since_food = 0
        self.steps = 0
        self.food = sample_food(self._rng, self.body_age)
        self._done = False
        return self._obs(), self._info(CAUSE_NONE)

    def step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict[str, Any]]:
        if self._done:
            raise RuntimeError("step() called on a finished episode; call reset() first")
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError(f"invalid action {action} for {self.action_space}")
        W = self.W

        # 1. heading and target cell
        heading = int(next_heading(self.heading, action, self.cfg.action_mode))
        head = self.head
        dr, dc = DELTAS[heading]
        new_head = (head[0] + int(dr), head[1] + int(dc))
        food_prev = self.food
        d_prev = int(manhattan(head, food_prev))
        d_moved = int(manhattan(new_head, food_prev))
        self.heading = heading
        self.steps += 1

        ate = died = False
        cause = CAUSE_NONE
        if not (0 <= new_head[0] < W and 0 <= new_head[1] < W):
            # 2. wall
            cause, died = CAUSE_WALL, True
        elif new_head == food_prev:
            # 3. eat: tail does not move
            ate = True
            self.length += 1
            self.body_age[new_head] = self.length
            self.score += 1
            self.steps_since_food = 0
            if self.length == W * W:
                cause = CAUSE_WIN
            else:
                self.food = sample_food(self._rng, self.body_age)
        else:
            # 4. move: vacate the tail first, then test the target cell
            self.body_age[self.body_age > 0] -= 1
            if self.body_age[new_head] > 0:
                cause, died = CAUSE_SELF, True
            else:
                self.body_age[new_head] = self.length
                self.steps_since_food += 1
                # 5. starvation truncation
                if self.steps_since_food >= self.H:
                    cause = CAUSE_STARVE

        terminated = cause in TERMINAL_CAUSES
        truncated = cause == CAUSE_STARVE
        self._done = terminated or truncated
        d_next = int(manhattan(new_head, self.food))
        reward = float(
            compute_reward(
                self.cfg.reward,
                W,
                ate=ate,
                died=died,
                terminal=terminated,
                d_prev=d_prev,
                d_moved=d_moved,
                d_next=d_next,
            )
        )
        return self._obs(), reward, terminated, truncated, self._info(cause)

    def render(self) -> str | None:
        if self.render_mode != "ansi":
            return None
        return self.to_ansi()

    # ------------------------------------------------------------ inspection
    @property
    def head(self) -> tuple[int, int]:
        r, c = np.unravel_index(int(np.argmax(self.body_age)), self.body_age.shape)
        return int(r), int(c)

    @property
    def rng(self) -> np.random.Generator:
        if self._rng is None:
            raise RuntimeError("call reset() first")
        return self._rng

    def snake_cells(self) -> list[tuple[int, int]]:
        """Cells tail -> head."""
        return cells_from_body_age(self.body_age)

    def to_ansi(self) -> str:
        chars = np.full((self.W, self.W), ".", dtype="<U1")
        chars[self.body_age > 0] = "o"
        chars[self.head] = "H"
        chars[self.food] = "*"
        return "\n".join("".join(row) for row in chars)

    def set_state(
        self,
        cells: Sequence[tuple[int, int]],
        food: tuple[int, int],
        heading: int | None = None,
        steps_since_food: int = 0,
    ) -> np.ndarray:
        """Overwrite the board (tests, GUI replay). ``cells`` are tail -> head.

        ``heading`` defaults to the neck -> head direction. Score and step
        counters are reset. Returns the observation.
        """
        if self._rng is None:
            self._rng = np.random.default_rng()
        body_age = body_age_from_cells(cells, self.W)
        for (r0, c0), (r1, c1) in zip(cells[:-1], cells[1:]):
            if abs(r0 - r1) + abs(c0 - c1) != 1:
                raise ValueError(f"cells {(r0, c0)} and {(r1, c1)} are not adjacent")
        if body_age[food] != 0:
            raise ValueError("food on the snake")
        self.body_age = body_age
        self.heading = infer_heading(body_age) if heading is None else int(heading)
        self.food = (int(food[0]), int(food[1]))
        self.length = len(cells)
        self.score = 0
        self.steps = 0
        self.steps_since_food = int(steps_since_food)
        self._done = False
        return self._obs()

    # -------------------------------------------------------------- helpers
    def _obs(self) -> np.ndarray:
        return encode(
            self.cfg.obs,
            self.body_age[None],
            np.array([self.heading]),
            np.array([self.food]),
            np.array([self.length]),
        )[0]

    def _info(self, cause: int) -> dict[str, Any]:
        return {
            "cause": CAUSE_NAMES[cause],
            "score": self.score,
            "length": self.length,
            "steps": self.steps,
        }

#### 2.5 The vectorised training environment

`N` boards advance together as one `(N, W, W)` batch of `body_age` grids; every rule is a masked array operation.
Sub-environment `i` behaves bitwise like a scalar `SnakeEnv` reset with `seed + i`. Finished sub-environments reset at
once: `step` returns the first observation of the new episode and keeps the true successor of the transition in
`info["final_obs"]`, which is what the learners store.

In [ ]:
# The vectorised training environment: N boards as one (N, W, W) batch, auto-reset, final_obs
from __future__ import annotations


class VecSnakeEnv:
    """``n_envs`` independent Snake boards stepped as one batch, with auto-reset.

    See the module docstring for the rules. ``debug=True`` runs
    :meth:`check_invariants` after every ``reset`` and ``step``.
    """

    def __init__(
        self,
        cfg: EnvConfig | None = None,
        n_envs: int = 128,
        seed: int = 0,
        debug: bool = False,
    ) -> None:
        self.cfg = cfg if cfg is not None else EnvConfig()
        if self.cfg.W < 4:
            raise ValueError("W must be at least 4")
        if n_envs < 1:
            raise ValueError("n_envs must be at least 1")
        self.n_envs = int(n_envs)
        self.W = self.cfg.W
        self.H = self.cfg.H
        self.n_actions = n_actions(self.cfg.action_mode)
        self.obs_shape = self.cfg.obs.shape(self.W)
        #: Base seed of the generators in use; sub-env ``i`` owns ``seed + i``.
        self.seed = int(seed)
        self.debug = bool(debug)

        N, W = self.n_envs, self.W
        self.body_age: np.ndarray = np.zeros((N, W, W), dtype=AGE_DTYPE)
        self.heading: np.ndarray = np.full(N, RIGHT, dtype=np.int64)
        self.food: np.ndarray = np.zeros((N, 2), dtype=np.int64)
        self.length: np.ndarray = np.zeros(N, dtype=np.int64)
        self.score: np.ndarray = np.zeros(N, dtype=np.int64)
        self.steps_since_food: np.ndarray = np.zeros(N, dtype=np.int64)
        self.steps: np.ndarray = np.zeros(N, dtype=np.int64)

        self._rngs = [np.random.default_rng(self.seed + i) for i in range(N)]
        #: Cached head cell, always equal to ``head_positions(body_age)``.
        self._head: np.ndarray = np.zeros((N, 2), dtype=np.int64)
        self._rows = np.arange(N)
        cells = initial_cells(W)
        self._start_grid = body_age_from_cells(cells, W)
        self._start_head = np.array(cells[-1], dtype=np.int64)
        self._start_length = len(cells)
        self._started = False

    # ------------------------------------------------------------------ API
    def reset(self, seed: int | None = None) -> np.ndarray:
        """Restart every board and return the observation ``(N, *obs_shape)``
        only (no info dict, unlike ``SnakeEnv.reset``).

        ``seed`` re-creates the generators as ``default_rng(seed + i)``;
        ``None`` keeps them, so the random streams continue.
        """
        if seed is not None:
            seed = int(seed)
            # Build first: an invalid seed must leave ``seed`` and the streams as they were.
            self._rngs = [np.random.default_rng(seed + i) for i in range(self.n_envs)]
            self.seed = seed
        self._restart(self._rows)
        self._started = True
        if self.debug:
            self.check_invariants()
        return self.observe()

    def step(
        self, actions: np.ndarray | Sequence[int]
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, dict[str, np.ndarray]]:
        """Advance every board by one action.

        Returns ``(obs, reward, terminated, truncated, info)``; all arrays have
        a leading ``N`` axis. ``obs`` is the observation to act on next (the
        first one of the new episode where the episode ended). ``info`` holds
        ``final_obs`` (true successor of this transition, before auto-reset),
        ``done`` (``terminated | truncated``), ``cause`` (``CAUSE_*`` ids) and
        ``score`` / ``length`` / ``steps`` as they stood before auto-reset.
        ``obs`` and ``info["final_obs"]`` are always two separate arrays, equal
        on the rows that did not finish.
        """
        self._require_reset()
        actions = self._check_actions(actions)
        W, rows, grid = self.W, self._rows, self.body_age

        # 1. heading and target cell
        heading = next_heading(self.heading, actions, self.cfg.action_mode)
        head = self._head
        new_head = head + DELTAS[heading]
        d_prev = manhattan(head, self.food)
        d_moved = manhattan(new_head, self.food)
        self.heading[:] = heading
        self.steps += 1

        # 2. wall / 3. eat masks. Food is always on the grid, so ``ate``
        # excludes ``wall``; everything else is an ordinary move.
        r, c = new_head[:, 0], new_head[:, 1]
        wall = (r < 0) | (r >= W) | (c < 0) | (c >= W)
        ate = (r == self.food[:, 0]) & (c == self.food[:, 1])
        move = ~(wall | ate)

        # 4. move: vacate the tail first, then test the target cell (so the
        # vacating tail cell is legal). Coordinates are clipped only for the
        # wall rows, whose lookup is masked out by ``move``.
        np.subtract(grid, 1, out=grid, where=move[:, None, None] & (grid > 0))
        occupied = grid[rows, np.clip(r, 0, W - 1), np.clip(c, 0, W - 1)] > 0
        hit = move & occupied
        ok = move & ~hit

        # Head placement (eat: with the new length; self collision: not placed).
        self.length += ate
        placed = np.flatnonzero(ate | ok)
        grid[placed, r[placed], c[placed]] = self.length[placed]
        head[placed] = new_head[placed]

        self.score += ate
        self.steps_since_food[ate] = 0
        self.steps_since_food += ok

        # 5. episode ends. The starvation check runs on plain moves only.
        win = ate & (self.length == W * W)
        starve = ok & (self.steps_since_food >= self.H)
        died = wall | hit
        terminated = died | win
        truncated = starve
        done = terminated | truncated
        cause = np.full(self.n_envs, CAUSE_NONE, dtype=np.int64)
        cause[wall] = CAUSE_WALL
        cause[hit] = CAUSE_SELF
        cause[starve] = CAUSE_STARVE
        cause[win] = CAUSE_WIN

        # New food for the eaters, unless the board is full.
        eaters = np.flatnonzero(ate & ~win)
        if eaters.size:
            self._spawn_food(eaters)
        d_next = manhattan(new_head, self.food)
        reward = compute_reward(
            self.cfg.reward,
            W,
            ate=ate,
            died=died,
            terminal=terminated,
            d_prev=d_prev,
            d_moved=d_moved,
            d_next=d_next,
        )

        # True successor of every transition, snapshotted before auto-reset.
        final_obs = self.observe()
        info = {
            "final_obs": final_obs,
            "done": done,
            "cause": cause,
            "score": self.score.copy(),
            "length": self.length.copy(),
            "steps": self.steps.copy(),
        }

        # 6. auto-reset: finished boards start their next episode now. ``obs``
        # is always its own array, so a caller editing it in place can never
        # corrupt the ``final_obs`` that goes into the replay buffer.
        obs = final_obs.copy()
        finished = np.flatnonzero(done)
        if finished.size:
            self._restart(finished)
            obs[finished] = encode(
                self.cfg.obs,
                grid[finished],
                self.heading[finished],
                self.food[finished],
                self.length[finished],
            )
        if self.debug:
            self.check_invariants()
        return obs, reward, terminated, truncated, info

    def observe(self) -> np.ndarray:
        """Fresh observation ``(N, *obs_shape)`` float32 of the current state."""
        self._require_reset()
        return encode(self.cfg.obs, self.body_age, self.heading, self.food, self.length)

    # ------------------------------------------------------------ inspection
    @property
    def head(self) -> np.ndarray:
        """``(N, 2)`` head cells ``(r, c)`` (a copy)."""
        return self._head.copy()

    def set_state(
        self,
        i: int,
        cells: Sequence[tuple[int, int]],
        food: tuple[int, int],
        heading: int | None = None,
        steps_since_food: int = 0,
    ) -> None:
        """Overwrite board ``i`` (tests, GUI replay). ``cells`` are tail -> head.

        Like ``SnakeEnv.set_state`` (``heading`` defaults to the neck -> head
        direction; score and step counters of board ``i`` are reset) but
        returns ``None``, requires ``reset()`` first and rejects off-grid
        cells. The other boards are untouched; a rejected call changes nothing.
        """
        self._require_reset()
        i = int(i)
        if not 0 <= i < self.n_envs:
            raise IndexError(f"sub-env index {i} out of range for n_envs = {self.n_envs}")
        W = self.W
        cells = [(int(r), int(c)) for r, c in cells]
        if not cells:
            raise ValueError("cells must not be empty")
        for r, c in cells:
            if not (0 <= r < W and 0 <= c < W):
                raise ValueError(f"cell {(r, c)} is off the grid")
        body_age = body_age_from_cells(cells, W)
        for (r0, c0), (r1, c1) in zip(cells[:-1], cells[1:]):
            if abs(r0 - r1) + abs(c0 - c1) != 1:
                raise ValueError(f"cells {(r0, c0)} and {(r1, c1)} are not adjacent")
        food = (int(food[0]), int(food[1]))
        if not (0 <= food[0] < W and 0 <= food[1] < W):
            raise ValueError(f"food {food} is off the grid")
        if body_age[food] != 0:
            raise ValueError("food on the snake")
        heading = infer_heading(body_age) if heading is None else int(heading)
        if not 0 <= heading <= 3:
            raise ValueError(f"invalid heading {heading}")
        steps_since_food = int(steps_since_food)  # may raise: convert before any write

        self.body_age[i] = body_age
        self.heading[i] = heading
        self.food[i] = food
        self.length[i] = len(cells)
        self.score[i] = 0
        self.steps[i] = 0
        self.steps_since_food[i] = steps_since_food
        self._head[i] = cells[-1]

    def check_invariants(self) -> None:
        """Raise ``AssertionError`` naming a broken sub-env (the lowest index
        that fails the first violated check).

        Checked for every board: the positive cells of ``body_age`` are exactly
        the ages ``1..length``, each once; consecutive ages are 4-adjacent;
        the food is on an empty in-bounds cell; ``heading`` is in ``0..3``;
        ``0 <= steps_since_food < H``; counters are non-negative; the cached
        head agrees with ``body_age``.
        """
        self._require_reset()
        N, W, rows = self.n_envs, self.W, self._rows
        n_cells = W * W
        flat = self.body_age.reshape(N, n_cells)
        length = self.length

        self._assert_all((length >= 1) & (length <= n_cells), "length outside 1..W*W")
        # Sorted in descending order a row must read length, ..., 2, 1, 0, ..., 0.
        expected = np.maximum(length[:, None] - np.arange(n_cells), 0)
        self._assert_all(
            (np.sort(flat, axis=1)[:, ::-1] == expected).all(axis=1),
            "body_age is not the ages 1..length, each exactly once",
        )
        # cell_of[n, k] = flat index of the cell with age k (column 0 is unused).
        cell_of = np.zeros((N, n_cells + 1), dtype=np.int64)
        cell_of[rows[:, None], flat] = np.arange(n_cells)
        r, c = np.divmod(cell_of[:, 1:], W)
        gap = np.abs(np.diff(r, axis=1)) + np.abs(np.diff(c, axis=1))  # ages k, k + 1
        linked = np.arange(2, n_cells + 1) <= length[:, None]  # age k + 1 exists
        self._assert_all(
            ~(linked & (gap != 1)).any(axis=1),
            "cells with consecutive ages are not adjacent",
        )

        food = self.food
        self._assert_all(((food >= 0) & (food < W)).all(axis=1), "food is off the grid")
        self._assert_all(self.body_age[rows, food[:, 0], food[:, 1]] == 0, "food is on the snake")
        self._assert_all((self.heading >= 0) & (self.heading <= 3), "heading outside 0..3")
        self._assert_all(
            (self.steps_since_food >= 0) & (self.steps_since_food < self.H),
            "steps_since_food outside 0..H-1",
        )
        self._assert_all((self.score >= 0) & (self.steps >= 0), "negative score or step counter")
        self._assert_all(
            (self._head == head_positions(self.body_age)).all(axis=1),
            "cached head disagrees with body_age",
        )

    # -------------------------------------------------------------- helpers
    def _require_reset(self) -> None:
        if not self._started:
            raise RuntimeError("call reset() first")

    def _check_actions(self, actions: np.ndarray | Sequence[int]) -> np.ndarray:
        actions = np.asarray(actions)
        if actions.shape != (self.n_envs,):
            raise ValueError(f"actions must have shape ({self.n_envs},), got {actions.shape}")
        if not np.issubdtype(actions.dtype, np.integer):
            raise ValueError(f"actions must be integers, got dtype {actions.dtype}")
        if actions.min() < 0 or actions.max() >= self.n_actions:
            raise ValueError(f"actions must lie in [0, {self.n_actions})")
        return actions.astype(np.int64, copy=False)

    def _restart(self, rows: np.ndarray) -> None:
        """Put boards ``rows`` into the initial state and draw their food."""
        self.body_age[rows] = self._start_grid
        self.heading[rows] = RIGHT
        self._head[rows] = self._start_head
        self.length[rows] = self._start_length
        self.score[rows] = 0
        self.steps_since_food[rows] = 0
        self.steps[rows] = 0
        self._spawn_food(rows)

    def _spawn_food(self, rows: np.ndarray) -> None:
        """Uniformly random empty cell for each board in ``rows``.

        Batched ``common.sample_food``: one key per cell from the board's own
        generator, occupied cells -> +inf, argmin. The caller guarantees an
        empty cell on every board in ``rows``.
        """
        keys = np.empty((rows.size, self.W * self.W))
        rngs = self._rngs
        for k, i in enumerate(rows.tolist()):  # one Generator per sub-env
            rngs[i].random(out=keys[k])
        keys[self.body_age[rows].reshape(rows.size, -1) > 0] = np.inf
        self.food[rows] = np.stack(np.divmod(keys.argmin(axis=1), self.W), axis=1)

    def _assert_all(self, ok: np.ndarray, what: str) -> None:
        if not ok.all():
            i = int(np.flatnonzero(~ok)[0])
            raise AssertionError(f"sub-env {i}: {what}")

#### 2.6 A first look at the environment

In [ ]:
# One 8 x 8 board under both observations, and the vectorised engine checked against the scalar one for 300 random steps on four boards
board_A = SnakeEnv(EnvConfig(W=8), render_mode="ansi")
obs_A, _ = board_A.reset(seed=0)
print(board_A.render(), "\n")
print("observation A:", obs_A.astype(int), "-> danger straight/left/right, heading one-hot, food up/down/left/right")
board_B = SnakeEnv(EnvConfig(W=8, obs=ObsConfig(kind="B")))
obs_B, _ = board_B.reset(seed=0)
print("observation B:", obs_B.shape, "planes (head, body, food, age, heading x 4); food plane:\n", obs_B[2].astype(int))

check_cfg = EnvConfig(W=8)
vec = VecSnakeEnv(check_cfg, n_envs=4, seed=100)
vec_obs = vec.reset()
scalars = [SnakeEnv(check_cfg) for _ in range(4)]
scalar_obs = np.stack([env.reset(seed=100 + i)[0] for i, env in enumerate(scalars)])
rng = np.random.default_rng(0)
mismatches = 0
for _ in range(300):
    actions = rng.integers(0, vec.n_actions, size=4)
    vec_obs, vec_reward, vec_term, vec_trunc, vec_info = vec.step(actions)
    for i, env in enumerate(scalars):
        s_obs, s_reward, s_term, s_trunc, _ = env.step(int(actions[i]))
        mismatches += int(not np.array_equal(s_obs, vec_info["final_obs"][i]) or s_reward != vec_reward[i] or s_term != vec_term[i])
        if s_term or s_trunc:
            s_obs, _ = env.reset()
        mismatches += int(not np.array_equal(s_obs, vec_obs[i]))
print("vectorised against scalar environment, 4 boards x 300 steps:", "identical" if mismatches == 0 else f"{mismatches} mismatches")

### 3. The Players (`snake_rl/agents`)

Every player implements the `Policy` protocol used by the evaluation: `act(obs, env) -> action` and `reset()`. The
reference players read the whole environment through `env`; the learned agents use only `obs`.

#### 3.1 The Policy protocol and the epsilon schedule

In [ ]:
# The Policy protocol every player implements: act(obs, env) -> action, reset()
from __future__ import annotations


@runtime_checkable
class Policy(Protocol):
    name: str

    def act(self, obs: np.ndarray, env: SnakeEnv) -> int: ...

    def reset(self) -> None:
        """Called at the start of every evaluation episode."""
        ...

In [ ]:
# The linear epsilon schedule, a pure function of the step counter
from __future__ import annotations


def linear_schedule(step, start: float, end: float, decay_steps: float) -> float:
    """Linear interpolation from ``start`` (at step 0) to ``end`` (at ``decay_steps``).

    ``start + (end - start) * min(1.0, step / decay_steps)`` for
    ``decay_steps > 0``; constant afterwards. ``decay_steps <= 0`` means "no
    decay phase" and gives ``end`` for every step.

    ``step`` is a number (the result is a Python ``float``) or a numpy array
    of steps (the result is a float64 array of the same shape).

    The value on the plateau is the formula evaluated at fraction 1.0, i.e.
    ``start + (end - start)``, which can differ from ``end`` in the last bit
    (1.0 -> 0.05 gives 0.05000000000000004). It is constant, so nothing
    downstream depends on the difference.
    """
    if isinstance(step, np.ndarray):
        if decay_steps <= 0:
            return np.full(step.shape, end, dtype=np.float64)
        return start + (end - start) * np.minimum(1.0, step / decay_steps)
    if decay_steps <= 0:
        return float(end)
    return float(start + (end - start) * min(1.0, step / decay_steps))

#### 3.2 Reference players

`RandomPolicy` is the floor. `BFSGreedyPolicy` follows the shortest path to the food on the grid (time-aware: a body
cell of age `k` is passable if the head arrives after at least `k` steps, the rule `step` applies), ignores
self-trapping and falls back to the first safe action. `HamiltonianPolicy` follows a fixed Hamiltonian cycle (even
`W`), never collides and never starves, so it wins every game: the upper bound in score and the lower bound in speed.

In [ ]:
# Reference players: random, BFS greedy (time-aware) and the Hamiltonian cycle
from __future__ import annotations

BASELINE_NAMES = ("random", "bfs", "hamiltonian")


class RandomPolicy:
    name = "random"

    def __init__(self, seed: int | None = 0) -> None:
        self._seed = seed
        self._rng = np.random.default_rng(seed)

    def reset(self) -> None:
        pass

    def act(self, obs: np.ndarray, env: SnakeEnv) -> int:
        return int(self._rng.integers(env.action_space.n))


# --------------------------------------------------------------------- BFS
def bfs_first_heading(body_age: np.ndarray, head: tuple[int, int], food: tuple[int, int]) -> int | None:
    """Heading of the first step of a shortest head -> food path, or ``None``.

    A cell reached at BFS depth ``d`` is passable if ``body_age <= d``: the
    tail vacates one cell per step and no food is eaten before the goal, so
    the occupancy at arrival is exactly the current age minus ``d``.
    """
    W = body_age.shape[0]
    dist = np.full((W, W), -1, dtype=np.int32)
    first = np.full((W, W), -1, dtype=np.int32)
    dist[head] = 0
    queue: deque[tuple[int, int]] = deque([head])
    while queue:
        r, c = queue.popleft()
        d = int(dist[r, c])
        if (r, c) == food:
            return int(first[r, c])
        for h in range(4):
            nr, nc = r + int(DELTAS[h, 0]), c + int(DELTAS[h, 1])
            if not (0 <= nr < W and 0 <= nc < W) or dist[nr, nc] != -1:
                continue
            if body_age[nr, nc] > d + 1:
                continue
            dist[nr, nc] = d + 1
            first[nr, nc] = h if d == 0 else first[r, c]
            queue.append((nr, nc))
    return None


def _is_safe(env: SnakeEnv, heading: int) -> bool:
    head = np.array([env.head])
    return not bool(cell_is_danger(env.body_age[None], head + DELTAS[heading])[0])


class BFSGreedyPolicy:
    name = "bfs"

    def reset(self) -> None:
        pass

    def act(self, obs: np.ndarray, env: SnakeEnv) -> int:
        desired = bfs_first_heading(env.body_age, env.head, env.food)
        if desired is None:
            for turn in RELATIVE_TURN:  # straight, left, right
                h = int((env.heading + turn) % 4)
                if _is_safe(env, h):
                    desired = h
                    break
            else:
                return STRAIGHT if env.cfg.action_mode == "relative" else int(env.heading)
        return heading_to_action(env.heading, desired, env.cfg.action_mode)


# ------------------------------------------------------------- Hamiltonian
def hamiltonian_cycle(W: int) -> np.ndarray:
    """``(W*W, 2)`` cells of a fixed Hamiltonian cycle on an even ``W`` grid.

    Column 0 is the return path (bottom -> top); rows sweep columns
    ``1..W-1`` alternately right / left starting at ``(0, 1)``. Consecutive
    entries are adjacent and the last entry ``(1, 0)`` is adjacent to the
    first ``(0, 0)``.
    """
    if W < 4 or W % 2:
        raise ValueError("a Hamiltonian cycle of this construction needs even W >= 4")
    cells: list[tuple[int, int]] = [(0, 0)]
    for r in range(W):
        cols = range(1, W) if r % 2 == 0 else range(W - 1, 0, -1)
        cells.extend((r, c) for c in cols)
    cells.extend((r, 0) for r in range(W - 1, 0, -1))
    return np.array(cells, dtype=np.int64)


class HamiltonianPolicy:
    """Follow the cycle in whichever direction does not run into the neck."""

    name = "hamiltonian"

    def __init__(self) -> None:
        self._W: int | None = None
        self._cycle: np.ndarray | None = None
        self._index: np.ndarray | None = None

    def reset(self) -> None:
        pass

    def _prepare(self, W: int) -> None:
        if self._W != W:
            self._cycle = hamiltonian_cycle(W)
            self._index = np.empty((W, W), dtype=np.int64)
            self._index[self._cycle[:, 0], self._cycle[:, 1]] = np.arange(W * W)
            self._W = W

    def act(self, obs: np.ndarray, env: SnakeEnv) -> int:
        self._prepare(env.W)
        assert self._cycle is not None and self._index is not None
        n = env.W * env.W
        head = env.head
        i = int(self._index[head])
        neck = (head[0] - int(DELTAS[env.heading, 0]), head[1] - int(DELTAS[env.heading, 1]))
        target = tuple(int(v) for v in self._cycle[(i + 1) % n])
        if target == neck:
            target = tuple(int(v) for v in self._cycle[(i - 1) % n])
        delta = (target[0] - head[0], target[1] - head[1])
        desired = next(h for h in range(4) if tuple(DELTAS[h].tolist()) == delta)
        return heading_to_action(env.heading, desired, env.cfg.action_mode)


def make_policy(name: str, seed: int | None = 0):
    if name == "random":
        return RandomPolicy(seed)
    if name == "bfs":
        return BFSGreedyPolicy()
    if name == "hamiltonian":
        return HamiltonianPolicy()
    raise ValueError(f"unknown baseline {name!r}; choose from {BASELINE_NAMES}")

#### 3.3 Tabular Q-learning on observation A

The reference point before function approximation: a state is the 11-bit integer of observation A, the table is
`q[2048, n_actions]`, and the update is `Q[s, a] += alpha * (r + gamma * max_a' Q[s', a'] - Q[s, a])` with the target
zero only when the transition `terminated` (a starvation truncation bootstraps). Training steps `N` boards at once,
so the update takes `N` transitions and moves every distinct cell by `alpha` times the mean TD error of its
transitions (duplicates are averaged, not summed). Exploration is epsilon-greedy from an explicit generator.

In [ ]:
# Tabular Q-learning on observation A: 11-bit state index, synchronous batch update, epsilon-greedy
from __future__ import annotations

#: Number of observation-A codes (rows of the table).
N_STATES_A = 2**N_FEATURES_A

#: Weight of feature ``k`` in the state index: ``2**k``. Floating point so
#: that the index is one matrix product; exact, because every partial sum is
#: an integer below 2048. The float32 copy avoids a cast of the environment's
#: float32 observations on the training hot path.
_BIT_WEIGHTS = 2.0 ** np.arange(N_FEATURES_A)
_BIT_WEIGHTS.setflags(write=False)
_BIT_WEIGHTS_F32 = _BIT_WEIGHTS.astype(np.float32)
_BIT_WEIGHTS_F32.setflags(write=False)


def state_index(obs: np.ndarray) -> np.ndarray:
    """Table row of observation(s) A: ``sum_k obs[..., k] * 2**k``.

    ``obs`` has shape ``(..., 11)`` and holds exactly 0 / 1 (any real or bool
    dtype; the values are not validated). Returns an int64 array of shape
    ``obs.shape[:-1]`` (0-d for a single observation).
    """
    obs = np.asarray(obs)
    if obs.ndim == 0 or obs.shape[-1] != N_FEATURES_A:
        raise ValueError(f"observation A must have a last axis of {N_FEATURES_A}, got shape {obs.shape}")
    weights = _BIT_WEIGHTS_F32 if obs.dtype == np.float32 else _BIT_WEIGHTS
    return np.asarray(obs @ weights).astype(np.int64)


@dataclass(frozen=True)
class TabularQConfig:
    #: Step size.
    alpha: float = 0.1
    #: Discount.
    gamma: float = 0.99
    eps_start: float = 1.0
    eps_end: float = 0.05
    #: Epsilon decays linearly over this fraction of ``train.total_env_steps``
    #: (used by the training loop, not by the agent).
    eps_decay_fraction: float = 0.2
    #: Initial value of every table entry.
    q_init: float = 0.0


class TabularQAgent:
    """Q-table over the 2048 observation-A codes. See the module docstring.

    ``q[s, a]`` are the action values (float64), ``visits[s, a]`` the number
    of transitions that updated each cell (int64). Both are public; the agent
    holds no other state and no RNG.
    """

    name = "tabular_q"

    def __init__(self, n_actions: int, cfg: TabularQConfig | None = None) -> None:
        if isinstance(n_actions, bool) or not isinstance(n_actions, (int, np.integer)) or n_actions < 1:
            raise ValueError(f"n_actions must be a positive integer, got {n_actions!r}")
        self.cfg = cfg if cfg is not None else TabularQConfig()
        self.n_actions = int(n_actions)
        self.q: np.ndarray = np.full((N_STATES_A, self.n_actions), self.cfg.q_init, dtype=np.float64)
        self.visits: np.ndarray = np.zeros((N_STATES_A, self.n_actions), dtype=np.int64)

    # ------------------------------------------------------------- acting
    def reset(self) -> None:
        """Policy protocol (start of an evaluation episode). Nothing to reset."""

    def act(self, obs: np.ndarray, env: SnakeEnv | None = None) -> int:
        """Greedy action for one observation ``(11,)`` (Policy protocol; ``env`` is unused)."""
        obs = np.asarray(obs)
        if obs.ndim != 1:
            raise ValueError(f"act() takes one observation of shape ({N_FEATURES_A},), got shape {obs.shape}")
        return int(np.argmax(self.q[state_index(obs)]))

    def act_batch(self, obs: np.ndarray) -> np.ndarray:
        """Greedy actions ``(N,)`` int64 for a batch of observations ``(N, 11)``."""
        obs = np.asarray(obs)
        if obs.ndim != 2:
            raise ValueError(f"act_batch() takes observations of shape (N, {N_FEATURES_A}), got shape {obs.shape}")
        return self.q[state_index(obs)].argmax(axis=1).astype(np.int64, copy=False)

    def act_epsilon(self, obs: np.ndarray, epsilon: float, rng: np.random.Generator) -> np.ndarray:
        """Epsilon-greedy actions ``(N,)`` int64 for a batch ``(N, 11)``.

        RNG protocol (fixed, so that runs are reproducible and tests can
        replicate the stream): first ``rng.random(N) < epsilon`` decides who
        explores, then ``rng.integers(0, n_actions, size=N)`` proposes a
        random action for everyone. Both draws happen on every call, in this
        order, whatever ``epsilon`` is.
        """
        greedy = self.act_batch(obs)
        n = greedy.shape[0]
        explore = rng.random(n) < epsilon
        random_actions = rng.integers(0, self.n_actions, size=n)
        return np.where(explore, random_actions, greedy)

    # ----------------------------------------------------------- learning
    def update(
        self,
        obs: np.ndarray,
        actions: np.ndarray,
        rewards: np.ndarray,
        next_obs: np.ndarray,
        terminated: np.ndarray,
    ) -> float:
        """One synchronous batch of Q-learning updates from ``N`` transitions.

        ``obs`` / ``next_obs`` are ``(N, 11)``; ``actions`` ``(N,)`` integers;
        ``rewards`` ``(N,)`` of any real dtype (the arithmetic is float64);
        ``terminated`` ``(N,)`` bool - the environment's ``terminated`` flag
        only. A starvation-truncated transition has ``terminated = False`` and
        bootstraps from ``next_obs``, which must be the true successor
        (``info["final_obs"]``), not the post-reset observation.

        With ``q`` as it stands before the call for every ``n``::

            target_n = r_n + gamma * (0 if terminated_n else max_a' q[s'_n, a'])
            td_n     = target_n - q[s_n, a_n]
            q[s, a] += alpha * mean(td_n over the transitions of cell (s, a))
            visits[s, a] += number of transitions of cell (s, a)

        Returns ``mean(|td_n|)`` over the batch.
        """
        obs = np.asarray(obs)
        next_obs = np.asarray(next_obs)
        actions = np.asarray(actions)
        rewards = np.asarray(rewards, dtype=np.float64)
        terminated = np.asarray(terminated, dtype=bool)
        n = actions.shape[0] if actions.ndim == 1 else -1
        if n < 1:
            raise ValueError(f"actions must have shape (N,) with N >= 1, got {actions.shape}")
        if not (
            obs.shape == next_obs.shape == (n, N_FEATURES_A)
            and rewards.shape == terminated.shape == (n,)
        ):
            raise ValueError(
                f"inconsistent batch for N = {n}: obs {obs.shape}, next_obs {next_obs.shape}, "
                f"rewards {rewards.shape}, terminated {terminated.shape}"
            )
        if not np.issubdtype(actions.dtype, np.integer):
            raise ValueError(f"actions must be integers, got dtype {actions.dtype}")
        if actions.min() < 0 or actions.max() >= self.n_actions:
            raise ValueError(f"actions must lie in [0, {self.n_actions})")

        q = self.q
        s = state_index(obs)
        next_max = q[state_index(next_obs)].max(axis=1)
        target = rewards + self.cfg.gamma * np.where(terminated, 0.0, next_max)
        q_sa = q[s, actions]
        td = target - q_sa

        # Group the transitions by cell (flat index s * n_actions + a): the
        # number of transitions and the mean TD error of each transition's cell.
        cell = s * self.n_actions + actions
        count = np.bincount(cell)[cell]
        td_mean = np.bincount(cell, weights=td)[cell] / count

        # One write. All transitions of a cell carry the same new value (old
        # value + alpha * mean TD error of the cell) and the same new visit
        # count, so the duplicate indices are harmless: each distinct cell
        # moves exactly once.
        q[s, actions] = q_sa + self.cfg.alpha * td_mean
        self.visits[s, actions] += count
        return float(np.abs(td).mean())

    # ---------------------------------------------------------------- I/O
    def save(self, path: str | Path) -> Path:
        """Write ``q``, ``visits``, ``n_actions`` and the config fields as a numpy
        ``.npz`` archive to exactly ``path`` (no suffix is appended)."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            np.savez(f, q=self.q, visits=self.visits, n_actions=np.int64(self.n_actions), **asdict(self.cfg))
        return path

    @classmethod
    def load(cls, path: str | Path) -> "TabularQAgent":
        """Exact inverse of :meth:`save` (``q``, ``visits`` and ``cfg`` round-trip)."""
        with np.load(Path(path)) as data:
            cfg = TabularQConfig(**{f.name: data[f.name].item() for f in fields(TabularQConfig)})
            agent = cls(int(data["n_actions"]), cfg)
            q, visits = data["q"], data["visits"]
        if q.shape != agent.q.shape or visits.shape != agent.visits.shape:
            raise ValueError(
                f"{path}: table shapes {q.shape} / {visits.shape} do not match "
                f"({N_STATES_A}, {agent.n_actions})"
            )
        agent.q = q.astype(np.float64, copy=False)
        agent.visits = visits.astype(np.int64, copy=False)
        return agent

#### 3.4 The DQN hyper-parameters

A torch-free dataclass of values (discount, learning rate, batch and buffer sizes, `learning_starts`, the replay
switch, `target_update_every`, gradient clipping, the epsilon schedule, the network layout, Double / Dueling / PER
switches, checkpoints, device); ranges are validated once, in the experiment configuration of Section 4.1.

In [ ]:
# DQNConfig: the hyper-parameters of the agent and of its training loop
from __future__ import annotations


@dataclass(frozen=True)
class DQNConfig:
    #: Discount.
    gamma: float = 0.99
    #: Adam step size.
    lr: float = 2.5e-4
    #: Transitions per gradient step (with replay).
    batch_size: int = 256
    #: Capacity of the replay buffer, in transitions (with replay).
    buffer_size: int = 100_000
    #: Experience replay (M6 ablation switch). ``False`` = no replay buffer:
    #: every vec-step after ``learning_starts`` is followed by exactly one
    #: gradient step on that vec-step's own ``n_envs`` transitions, each used
    #: once; ``batch_size``, ``buffer_size`` and ``grad_steps_per_vec_step``
    #: are then unused (the latter must be null or 1).
    replay: bool = True
    #: Environment steps collected before the first gradient step.
    learning_starts: int = 10_000
    #: Gradient steps after every vec-step (``n_envs`` environment steps).
    #: ``None`` means ``max(1, n_envs // 4)``: one gradient step per four
    #: environment steps, the update-to-data ratio of Mnih et al. (2015).
    grad_steps_per_vec_step: int | None = None
    #: Gradient steps between two hard target syncs. 1 = the target network is
    #: synchronised after every step, i.e. no separate target network in effect.
    target_update_every: int = 1_000
    #: Maximum global gradient norm (clipping threshold).
    grad_clip: float = 10.0
    eps_start: float = 1.0
    eps_end: float = 0.05
    #: Epsilon decays linearly over this fraction of ``train.total_env_steps``
    #: (used by the training loop, not by the agent).
    eps_decay_fraction: float = 0.2
    #: Widths of the fully connected hidden layers: the whole network on
    #: observation A (``()`` = a linear Q-function), the layers after the
    #: convolutions on observation B.
    hidden: tuple[int, ...] = (256, 256)
    #: Observation B only: output channels of the 3x3 convolution layers of
    #: the CNN (``networks.CNN``), in order. ``()`` = no convolution, the grid
    #: is flattened into the MLP of ``hidden``. Ignored on observation A.
    conv_channels: tuple[int, ...] = (32, 64, 64)
    #: Double DQN target (van Hasselt et al. 2016); studied in M7.
    double: bool = False
    #: Dueling head (Wang et al. 2016): ``Q = V + A - mean(A)`` with separate
    #: value and advantage streams after the shared layers (``networks.DuelingMLP``).
    dueling: bool = False
    #: Prioritised experience replay (Schaul et al. 2016, proportional variant):
    #: transitions are sampled with probability proportional to their priority
    #: ``(|td| + per_eps) ** per_alpha`` and the loss is importance-weighted.
    #: Needs ``replay``.
    per: bool = False
    #: Priority exponent alpha (0 = uniform replay with weights of one).
    per_alpha: float = 0.6
    #: Importance-sampling exponent beta at step 0; the training loop anneals it
    #: linearly to 1.0 at ``train.total_env_steps`` (full correction at the end).
    per_beta_start: float = 0.4
    #: Added to |td| before the exponent, so that no transition has priority 0.
    per_eps: float = 1e-6
    #: Training loop: write a weights-only checkpoint at every evaluation.
    keep_checkpoints: bool = True
    #: ``"auto"`` (cuda if available, else cpu) | ``"cpu"`` | ``"cuda"``.
    device: str = "auto"
    #: Training loop: ``torch.set_num_threads`` at trainer start; ``None``
    #: leaves torch's setting as it is.
    torch_threads: int | None = None

    def __post_init__(self) -> None:
        # A YAML file or a checkpoint delivers ``hidden`` / ``conv_channels``
        # as lists; the frozen (hashable, comparable) config keeps tuples.
        # Anything else is left as it is for the validation in
        # ``snake_rl.config`` to report.
        for name in ("hidden", "conv_channels"):
            value = getattr(self, name)
            if isinstance(value, list):
                object.__setattr__(self, name, tuple(value))

#### 3.5 Experience replay

`ReplayBuffer`: a fixed-size FIFO store of `(s, a, r, s', terminated)` in preallocated numpy arrays, sampled
uniformly from a generator the caller passes in, so a batch mixes many boards and many past policies.
`PrioritizedReplayBuffer` (Schaul et al. 2016, proportional variant): priority `(|td| + eps) ** alpha`, stratified
sampling, importance-sampling weights `(n P(i)) ** (-beta)` normalised to at most 1, `beta` annealed to 1 by the
training loop; a cumulative sum over the priorities instead of a sum tree.

In [ ]:
# Experience replay: the uniform FIFO buffer and the prioritised buffer with importance weights
from __future__ import annotations


@dataclass(frozen=True)
class Batch:
    """``B`` transitions as numpy arrays (copies, independent of the buffer)."""

    #: float32 ``(B, *obs_shape)``
    obs: np.ndarray
    #: int64 ``(B,)``
    actions: np.ndarray
    #: float32 ``(B,)``
    rewards: np.ndarray
    #: float32 ``(B, *obs_shape)``: the true successor observation.
    next_obs: np.ndarray
    #: bool ``(B,)``: the environment's ``terminated`` flag (never ``done``).
    terminated: np.ndarray


class ReplayBuffer:
    """Ring buffer of the most recent ``capacity`` transitions. See the module docstring."""

    def __init__(self, capacity: int, obs_shape: tuple[int, ...]) -> None:
        if isinstance(capacity, bool) or not isinstance(capacity, (int, np.integer)) or capacity < 1:
            raise ValueError(f"capacity must be an integer >= 1, got {capacity!r}")
        self.capacity = int(capacity)
        self.obs_shape = tuple(int(d) for d in obs_shape)
        self._obs = np.zeros((self.capacity, *self.obs_shape), dtype=np.float32)
        self._actions = np.zeros(self.capacity, dtype=np.int64)
        self._rewards = np.zeros(self.capacity, dtype=np.float32)
        self._next_obs = np.zeros((self.capacity, *self.obs_shape), dtype=np.float32)
        self._terminated = np.zeros(self.capacity, dtype=bool)
        self._pos = 0  # slot of the next write = slot of the oldest transition once full
        self._size = 0

    def __len__(self) -> int:
        """Number of stored transitions (at most ``capacity``)."""
        return self._size

    def add(
        self,
        obs: np.ndarray,
        actions: np.ndarray,
        rewards: np.ndarray,
        next_obs: np.ndarray,
        terminated: np.ndarray,
    ) -> None:
        """Append ``N`` transitions (leading axis ``N``), copying them into the storage.

        ``obs`` / ``next_obs`` are ``(N, *obs_shape)``, ``actions`` /
        ``rewards`` / ``terminated`` are ``(N,)``. The caller keeps ownership
        of its arrays and may reuse them. Once the buffer is full the oldest
        transitions are overwritten; if ``N > capacity`` only the last
        ``capacity`` transitions are kept. ``ValueError`` on inconsistent
        shapes.
        """
        obs = np.asarray(obs)
        next_obs = np.asarray(next_obs)
        actions = np.asarray(actions)
        rewards = np.asarray(rewards)
        terminated = np.asarray(terminated)
        n = actions.shape[0] if actions.ndim == 1 else -1
        if not (
            n >= 0
            and obs.shape == next_obs.shape == (n, *self.obs_shape)
            and rewards.shape == terminated.shape == (n,)
        ):
            raise ValueError(
                f"inconsistent transitions for obs_shape {self.obs_shape}: obs {obs.shape}, actions {actions.shape}, "
                f"rewards {rewards.shape}, next_obs {next_obs.shape}, terminated {terminated.shape}"
            )

        # Equivalent to N single appends: if the batch is longer than the ring,
        # its first N - capacity transitions would be overwritten by its own tail.
        skip = max(0, n - self.capacity)
        start = (self._pos + skip) % self.capacity
        count = n - skip
        first = min(count, self.capacity - start)  # transitions that fit before the end of the storage
        fields = (
            (self._obs, obs),
            (self._actions, actions),
            (self._rewards, rewards),
            (self._next_obs, next_obs),
            (self._terminated, terminated),
        )
        for store, values in fields:
            store[start : start + first] = values[skip : skip + first]
            store[: count - first] = values[skip + first :]  # the part that wraps around (possibly empty)
        self._pos = (start + count) % self.capacity
        self._size = min(self.capacity, self._size + n)

    def sample(self, batch_size: int, rng: np.random.Generator) -> Batch:
        """``batch_size`` transitions drawn uniformly **with** replacement.

        RNG protocol (fixed): exactly one call
        ``idx = rng.integers(0, len(self), size=batch_size)``; ``idx`` indexes
        the storage slots, which hold the transitions in insertion order until
        the buffer wraps around. ``ValueError`` if the buffer is empty.
        """
        if self._size == 0:
            raise ValueError("cannot sample from an empty replay buffer")
        idx = rng.integers(0, self._size, size=batch_size)
        return Batch(*self._gather(idx))

    def _gather(self, idx: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """The five transition arrays of the slots ``idx`` (fancy indexing copies)."""
        return self._obs[idx], self._actions[idx], self._rewards[idx], self._next_obs[idx], self._terminated[idx]


@dataclass(frozen=True)
class PrioritizedBatch(Batch):
    """A :class:`Batch` drawn by priority, with what the trainer needs to
    weight the loss and to write the new priorities back."""

    #: int64 ``(B,)``: the storage slots the transitions were drawn from.
    indices: np.ndarray
    #: float32 ``(B,)``: importance-sampling weights, each in ``(0, 1]``.
    weights: np.ndarray


class PrioritizedReplayBuffer(ReplayBuffer):
    """Ring buffer with proportional prioritisation. See the module docstring.

    ``alpha`` is the priority exponent (0 = uniform sampling, every weight
    1), ``eps`` the constant added to ``|td|`` so no priority is 0.
    ``max_priority`` is the largest priority ever assigned (1.0 at the start):
    the priority of a newly added transition.
    """

    def __init__(self, capacity: int, obs_shape: tuple[int, ...], alpha: float = 0.6, eps: float = 1e-6) -> None:
        super().__init__(capacity, obs_shape)
        if not (isinstance(alpha, (int, float)) and not isinstance(alpha, bool) and 0.0 <= alpha <= 1.0):
            raise ValueError(f"alpha must be a number in [0, 1], got {alpha!r}")
        if not (isinstance(eps, (int, float)) and not isinstance(eps, bool) and eps > 0.0):
            raise ValueError(f"eps must be a positive number, got {eps!r}")
        self.alpha = float(alpha)
        self.eps = float(eps)
        self.max_priority = 1.0
        self._priority = np.zeros(self.capacity, dtype=np.float64)

    @property
    def priorities(self) -> np.ndarray:
        """Copy of the priorities of the stored slots ``(len(self),)``, in slot order."""
        return self._priority[: self._size].copy()

    def add(
        self,
        obs: np.ndarray,
        actions: np.ndarray,
        rewards: np.ndarray,
        next_obs: np.ndarray,
        terminated: np.ndarray,
    ) -> None:
        """As :meth:`ReplayBuffer.add`; every slot written gets priority ``max_priority``."""
        pos_before = self._pos
        super().add(obs, actions, rewards, next_obs, terminated)  # validates the shapes, writes the ring
        # The same slots the base class wrote: the last min(N, capacity)
        # transitions, from the (skipped-ahead) old write position onwards.
        n = np.asarray(actions).shape[0]
        skip = max(0, n - self.capacity)
        slots = ((pos_before + skip) + np.arange(n - skip)) % self.capacity
        self._priority[slots] = self.max_priority

    def sample(self, batch_size: int, rng: np.random.Generator, beta: float) -> PrioritizedBatch:  # type: ignore[override]
        """``batch_size`` transitions drawn by priority, stratified.

        RNG protocol (fixed): exactly one call ``u = rng.random(batch_size)``.
        Sample ``k`` is the slot whose cumulative priority mass covers
        ``(k + u_k) / batch_size * total``. ``weights`` are
        ``(p_i / p_min) ** (-beta)``, i.e. ``(n P(i)) ** (-beta)`` divided by
        its maximum over the stored slots. ``ValueError`` if the buffer is
        empty or ``beta`` is outside ``[0, 1]``.
        """
        if self._size == 0:
            raise ValueError("cannot sample from an empty replay buffer")
        if not (isinstance(beta, (int, float)) and not isinstance(beta, bool) and 0.0 <= beta <= 1.0):
            raise ValueError(f"beta must be a number in [0, 1], got {beta!r}")
        p = self._priority[: self._size]
        total = float(p.sum())
        u = rng.random(batch_size)
        mass = (np.arange(batch_size) + u) / batch_size * total
        idx = np.minimum(np.searchsorted(np.cumsum(p), mass, side="right"), self._size - 1)
        weights = (p[idx] / p.min()) ** (-float(beta))
        return PrioritizedBatch(
            *self._gather(idx),
            indices=idx.astype(np.int64, copy=False),
            weights=weights.astype(np.float32),
        )

    def update_priorities(self, indices: np.ndarray, td_abs: np.ndarray) -> None:
        """Set the priority of the slots ``indices`` to ``(|td_abs| + eps) ** alpha``.

        Duplicate indices: the last value wins. ``ValueError`` for arrays that
        are not 1-D of the same shape, an index outside the stored slots, or
        a non-finite ``td_abs``.
        """
        indices = np.asarray(indices)
        td_abs = np.asarray(td_abs, dtype=np.float64)
        if indices.ndim != 1 or indices.shape != td_abs.shape:
            raise ValueError(f"indices and td_abs must be 1-D of the same shape, got {indices.shape} and {td_abs.shape}")
        if indices.size and (indices.min() < 0 or indices.max() >= self._size):
            raise ValueError(f"indices must lie in 0..{self._size - 1} (the stored slots), got {indices.tolist()}")
        if not np.isfinite(td_abs).all():
            raise ValueError(f"td_abs must be finite, got {td_abs.tolist()}")
        p = (np.abs(td_abs) + self.eps) ** self.alpha
        self._priority[indices] = p
        if p.size:
            self.max_priority = max(self.max_priority, float(p.max()))

#### 3.6 The Q-networks

`MLP` on the 11 features of observation A, `CNN` on the `(C, W, W)` planes of observation B, and the `DuelingMLP`
head `Q(s, a) = V(s) + A(s, a) - mean_a' A(s, a')` (Wang et al. 2016), used in place of the plain head on A and as
the fully connected part of the CNN on B. `build_q_network` picks the architecture from the observation shape, so the
agent never names one.

In [ ]:
# Q-networks: the MLP on observation A, the CNN on observation B, the Dueling head
from __future__ import annotations


class MLP(nn.Module):
    """Fully connected Q-network.

    ``Linear -> ReLU`` for every width in ``hidden``, then a ``Linear`` layer
    to ``n_actions`` without activation (action values are unbounded).
    ``hidden = ()`` gives a linear Q-function ``Q(s, .) = W s + b``. The
    parameters take torch's default initialisation, drawn from the global CPU
    generator at construction (the agent seeds that draw).
    """

    def __init__(self, in_features: int, n_actions: int, hidden: Sequence[int]) -> None:
        super().__init__()
        widths = [int(in_features), *(int(h) for h in hidden)]
        if min(widths) < 1 or n_actions < 1:
            raise ValueError(
                f"layer sizes must be positive: in_features {in_features}, hidden {tuple(hidden)}, n_actions {n_actions}"
            )
        layers: list[nn.Module] = []
        for fan_in, fan_out in zip(widths[:-1], widths[1:]):
            layers += [nn.Linear(fan_in, fan_out), nn.ReLU()]
        layers.append(nn.Linear(widths[-1], int(n_actions)))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """``x`` float32 ``(N, in_features)`` -> action values ``(N, n_actions)``."""
        return self.net(x)


class DuelingMLP(nn.Module):
    """Fully connected Q-network with the Dueling head of Wang et al. (2016).

    # NEW: no resource equivalent - the course notebooks have no dueling
    # architecture (docs/RESOURCE_BASELINE.md, M7 row).

    The action value is decomposed into a state value and action advantages::

        Q(s, a) = V(s) + A(s, a) - mean_a' A(s, a')

    computed by two streams that share the first layers. ``hidden[:-1]`` are
    the shared ``Linear -> ReLU`` layers (``self.shared``, the identity when
    ``hidden`` has at most one width); each stream then has its own
    ``Linear(trunk, hidden[-1]) -> ReLU`` and an output layer, ``self.value``
    to one number and ``self.advantage`` to ``n_actions`` (with ``hidden =
    ()`` each stream is a single linear map). The mean subtraction is eq. 9
    of the paper: without it ``V`` and ``A`` are not identifiable (a constant
    can move between them), with it ``A`` has zero mean over the actions and
    ``V`` is the mean action value. The greedy action is the argmax of ``A``.

    The point of the split: the state value is updated by every transition
    from a state, whatever action it stored, so states whose value does not
    depend on the action (most Snake states: only a few are a choice between
    life and death) are learned with fewer samples. M7 measures whether that
    shows on this task. Parameters take torch's default initialisation from
    the global CPU generator, in the order shared, value, advantage.
    """

    def __init__(self, in_features: int, n_actions: int, hidden: Sequence[int]) -> None:
        super().__init__()
        widths = [int(w) for w in hidden]
        if int(in_features) < 1 or any(w < 1 for w in widths) or int(n_actions) < 1:
            raise ValueError(
                f"layer sizes must be positive: in_features {in_features}, hidden {tuple(hidden)}, n_actions {n_actions}"
            )
        shared: list[nn.Module] = []
        previous = int(in_features)
        for width in widths[:-1]:
            shared += [nn.Linear(previous, width), nn.ReLU()]
            previous = width
        self.shared = nn.Sequential(*shared)
        trunk = previous  # hidden[-2], or in_features without shared layers
        self.value = self._stream(trunk, widths, 1)
        self.advantage = self._stream(trunk, widths, int(n_actions))

    @staticmethod
    def _stream(trunk: int, widths: list[int], out_features: int) -> nn.Sequential:
        if widths:
            return nn.Sequential(nn.Linear(trunk, widths[-1]), nn.ReLU(), nn.Linear(widths[-1], out_features))
        return nn.Sequential(nn.Linear(trunk, out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """``x`` float32 ``(N, in_features)`` -> action values ``(N, n_actions)``."""
        h = self.shared(x)
        advantage = self.advantage(h)
        return self.value(h) + advantage - advantage.mean(dim=1, keepdim=True)


class CNN(nn.Module):
    """Convolutional Q-network for the grid observation B.

    Adapted from ``resources/week09/43008-Week9-Part-D-DQN-Atari-CustomCNN-
    Solution-Colab.ipynb`` cell 13 (``CustomCNN``): a ``Sequential`` stack of
    ``Conv2d -> ReLU`` ending in ``Flatten``, the flattened size found by one
    dummy forward pass, then the fully connected part. Changed for this
    repository (docs/RESOURCE_BASELINE.md section 2): the base class is
    ``nn.Module`` (no stable-baselines3); the convolutions are 3x3, stride 1,
    padding 1 (the resource's Atari kernels 8x8/4, 4x4/2 leave nothing of a
    10x10 board; padding keeps the board size, so ``n_flatten`` is
    ``conv_channels[-1] * W * W``); the dummy input is ``torch.zeros(1, *in_shape)``
    (there is no ``observation_space`` here); and the resource's
    ``Linear(n_flatten, features_dim) -> ReLU`` plus the Q head
    ``Linear(features_dim, n_actions)`` that SB3 adds behind it are this
    repository's :class:`MLP` with ``hidden = (features_dim,)`` (the shipped
    configurations use the resource's ``features_dim = 512``).

    ``conv_channels = ()`` gives no convolution at all: ``Flatten`` followed
    by the MLP, i.e. a fully connected network on the raw grid (an ablation
    of the convolutional prior). With ``dueling`` the fully connected part is
    a :class:`DuelingMLP` instead (M7). Parameters take torch's default
    initialisation from the global CPU generator, as in :class:`MLP`.
    """

    def __init__(
        self,
        in_shape: tuple[int, int, int],
        n_actions: int,
        conv_channels: Sequence[int],
        hidden: Sequence[int],
        dueling: bool = False,
    ) -> None:
        super().__init__()
        shape = tuple(int(d) for d in in_shape)
        if len(shape) != 3 or min(shape) < 1:
            raise ValueError(f"in_shape must be three positive integers (C, H, W), got {tuple(in_shape)}")
        channels = [int(c) for c in conv_channels]
        if any(c < 1 for c in channels) or n_actions < 1:
            raise ValueError(f"conv_channels must be positive, got {tuple(conv_channels)}; n_actions {n_actions}")
        layers: list[nn.Module] = []
        previous = shape[0]
        for c in channels:
            layers += [nn.Conv2d(previous, c, kernel_size=3, stride=1, padding=1), nn.ReLU()]
            previous = c
        self.cnn = nn.Sequential(*layers, nn.Flatten())
        # Compute the size of the output tensor after passing through the CNN
        # (resource cell 13, with a zero tensor in place of observation_space.sample()).
        with torch.no_grad():
            self.n_flatten = int(self.cnn(torch.zeros(1, *shape)).shape[1])
        head = DuelingMLP if dueling else MLP
        self.head = head(self.n_flatten, n_actions, hidden)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """``x`` float32 ``(N, C, H, W)`` -> action values ``(N, n_actions)``."""
        return self.head(self.cnn(x))


def build_q_network(
    obs_shape: tuple[int, ...],
    n_actions: int,
    hidden: Sequence[int],
    conv_channels: Sequence[int] = (32, 64, 64),
    dueling: bool = False,
) -> nn.Module:
    """The Q-network for observations of shape ``obs_shape``.

    A 1-D shape (observation A) gives an :class:`MLP` (a :class:`DuelingMLP`
    with ``dueling``) and ignores ``conv_channels``; a 3-D shape ``(C, H, W)``
    (observation B) gives a :class:`CNN`, whose fully connected part is a
    dueling head with ``dueling``. Any other shape is a ``ValueError``.
    """
    shape = tuple(obs_shape)
    if len(shape) == 1:
        return (DuelingMLP if dueling else MLP)(shape[0], n_actions, hidden)
    if len(shape) == 3:
        return CNN(shape, n_actions, conv_channels, hidden, dueling=dueling)
    raise ValueError(
        f"no Q-network for observations of shape {shape}: expected (features,) for observation A "
        f"or (channels, height, width) for observation B"
    )

#### 3.7 The DQN agent

Online network `theta` and target network `theta^-` of the same architecture, Adam. One gradient step on a batch:
`target = r + gamma * (0 if terminated else max_a' Q(s', a'; theta^-))`, Huber loss of `Q(s, a; theta) - target`,
global gradient norm clipped; the target is a constant (semi-gradient), and `theta^-` is copied from `theta` every
`target_update_every` gradient steps. Double DQN selects `a*` with the online network and values it with the target
network; prioritised replay passes importance weights into the loss and gets the new TD errors back as priorities.
Greedy ties go to the lowest action index; `act` is `act_batch` on a batch of one.

In [ ]:
# DQNAgent: acting, the Huber gradient step, Double DQN target, target network, checkpoints
from __future__ import annotations


def resolve_device(name: str) -> torch.device:
    """``"auto"`` -> cuda if available, else cpu; ``"cpu"`` / ``"cuda"`` as named."""
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if name == "cpu":
        return torch.device("cpu")
    if name == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError('device "cuda" requested but torch.cuda.is_available() is False')
        return torch.device("cuda")
    raise ValueError(f'device must be "auto", "cpu" or "cuda", got {name!r}')


class DQNAgent:
    """DQN with a hard-synchronised target network. See the module docstring.

    Public state: ``q_net`` (online), ``target_net``, ``optimizer`` and
    ``grad_steps``, the number of gradient steps taken so far.
    """

    name = "dqn"

    def __init__(
        self,
        obs_shape: tuple[int, ...],
        n_actions: int,
        cfg: DQNConfig | None = None,
        seed: int = 0,
    ) -> None:
        if isinstance(n_actions, bool) or not isinstance(n_actions, (int, np.integer)) or n_actions < 1:
            raise ValueError(f"n_actions must be a positive integer, got {n_actions!r}")
        self.cfg = cfg if cfg is not None else DQNConfig()
        self.obs_shape = tuple(int(d) for d in obs_shape)
        self.n_actions = int(n_actions)
        self.device = resolve_device(self.cfg.device)

        # The initial weights are a function of ``seed`` alone: built on the CPU
        # from the freshly seeded default generator, whose state is restored
        # afterwards, so creating an agent leaves the global torch RNG untouched.
        # (Seeding the generator object, not torch.manual_seed, keeps the CUDA
        # generators out of it.)
        with torch.random.fork_rng(devices=[]):
            torch.default_generator.manual_seed(int(seed))
            q_net = build_q_network(
                self.obs_shape, self.n_actions, self.cfg.hidden, self.cfg.conv_channels, dueling=self.cfg.dueling
            )
        self.q_net = q_net.to(self.device)
        self.target_net = copy.deepcopy(self.q_net)
        self.target_net.requires_grad_(False)  # only ever written by a copy of q_net
        self.optimizer = torch.optim.Adam(
            self.q_net.parameters(), lr=self.cfg.lr, fused=self.device.type == "cuda"
        )
        self.grad_steps = 0

    # ------------------------------------------------------------- acting
    def reset(self) -> None:
        """Policy protocol (start of an evaluation episode). Nothing to reset."""

    def q_values(self, obs: np.ndarray) -> np.ndarray:
        """Action values ``(N, n_actions)`` float32 of the online network for
        observations ``(N, *obs_shape)``. No gradient is recorded."""
        obs = np.asarray(obs)
        if obs.ndim != len(self.obs_shape) + 1 or obs.shape[1:] != self.obs_shape:
            raise ValueError(f"q_values() takes observations of shape (N, {self._shape_text()}), got shape {obs.shape}")
        with torch.no_grad():
            q = self.q_net(torch.as_tensor(obs, dtype=torch.float32, device=self.device))
        return q.cpu().numpy()

    def act(self, obs: np.ndarray, env: SnakeEnv | None = None) -> int:
        """Greedy action for one observation ``obs_shape`` (Policy protocol; ``env`` is unused)."""
        obs = np.asarray(obs)
        if obs.shape != self.obs_shape:
            raise ValueError(f"act() takes one observation of shape ({self._shape_text()}), got shape {obs.shape}")
        return int(self.act_batch(obs[None])[0])

    def act_batch(self, obs: np.ndarray) -> np.ndarray:
        """Greedy actions ``(N,)`` int64 for observations ``(N, *obs_shape)``;
        ties go to the lowest action index."""
        return self.q_values(obs).argmax(axis=1).astype(np.int64, copy=False)

    def act_epsilon(self, obs: np.ndarray, epsilon: float, rng: np.random.Generator) -> np.ndarray:
        """Epsilon-greedy actions ``(N,)`` int64 for observations ``(N, *obs_shape)``.

        RNG protocol (fixed, the one of ``TabularQAgent.act_epsilon``): first
        ``rng.random(N) < epsilon`` decides who explores, then
        ``rng.integers(0, n_actions, size=N)`` proposes a random action for
        everyone. Both draws happen on every call, in this order, whatever
        ``epsilon`` is.
        """
        greedy = self.act_batch(obs)
        n = greedy.shape[0]
        explore = rng.random(n) < epsilon
        random_actions = rng.integers(0, self.n_actions, size=n)
        return np.where(explore, random_actions, greedy)

    # ----------------------------------------------------------- learning
    def update(self, batch: Batch, weights: np.ndarray | None = None) -> dict[str, Any]:
        """One gradient step on a batch of transitions (module docstring).

        ``batch.terminated`` is the environment's ``terminated`` flag only: a
        starvation-truncated transition has ``terminated = False`` and
        bootstraps from ``batch.next_obs``, which must be the true successor
        (``info["final_obs"]``).

        ``weights`` (``(B,)`` floats, prioritised replay) multiply each
        transition's Huber loss before the mean; ``None`` means all ones and
        gives exactly the unweighted step.

        Returns ``loss`` (the weighted loss that was minimised), ``mean_q``
        (mean of ``q_sa``), ``mean_target`` and ``grad_norm`` (global
        gradient norm before clipping) as Python floats, and, only when
        ``weights`` is given, ``td_abs``: ``|q_sa - target|`` per transition
        before the step, a float32 array ``(B,)`` (the new priorities' input).
        A non-finite loss or gradient norm raises ``FloatingPointError``
        before the optimiser step: the networks, the optimiser state and
        ``grad_steps`` are left as they were.
        """
        cfg, device = self.cfg, self.device
        obs = torch.as_tensor(batch.obs, dtype=torch.float32, device=device)
        actions = torch.as_tensor(batch.actions, dtype=torch.int64, device=device)
        rewards = torch.as_tensor(batch.rewards, dtype=torch.float32, device=device)
        next_obs = torch.as_tensor(batch.next_obs, dtype=torch.float32, device=device)
        terminated = torch.as_tensor(batch.terminated, dtype=torch.bool, device=device)
        if weights is not None:
            weights = np.asarray(weights)
            if weights.shape != (actions.shape[0],):
                raise ValueError(f"weights must have shape ({actions.shape[0]},), got {weights.shape}")
            weights = torch.as_tensor(weights, dtype=torch.float32, device=device)

        with torch.no_grad():  # the target is a constant of the regression
            next_q = self.target_net(next_obs)
            if cfg.double:
                # Double DQN: the online network selects, the target network evaluates.
                best = self.q_net(next_obs).argmax(dim=1, keepdim=True)
                next_value = next_q.gather(1, best).squeeze(1)
            else:
                next_value = next_q.max(dim=1).values
            # Bootstrap unless terminated; a truncated transition keeps its future value.
            target = rewards + cfg.gamma * next_value.masked_fill(terminated, 0.0)

        q_sa = self.q_net(obs).gather(1, actions.unsqueeze(1)).squeeze(1)
        if weights is None:
            loss = F.smooth_l1_loss(q_sa, target, beta=1.0)  # Huber, mean over the batch
        else:  # importance-weighted Huber (prioritised replay)
            loss = (weights * F.smooth_l1_loss(q_sa, target, beta=1.0, reduction="none")).mean()

        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), cfg.grad_clip)

        # One host transfer for all the statistics, taken before the step so
        # that a NaN / inf aborts with the weights still intact.
        stats = torch.stack((loss.detach(), q_sa.detach().mean(), target.mean(), grad_norm))
        if weights is not None:
            stats = torch.cat((stats, (q_sa.detach() - target).abs()))
        values = stats.tolist()
        loss_value, mean_q, mean_target, norm_value = values[:4]
        if not (math.isfinite(loss_value) and math.isfinite(norm_value)):
            raise FloatingPointError(
                f"non-finite DQN update at gradient step {self.grad_steps + 1}: "
                f"loss {loss_value}, gradient norm {norm_value}"
            )
        self.optimizer.step()
        self.grad_steps += 1
        if self.grad_steps % cfg.target_update_every == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())
        result: dict[str, Any] = {"loss": loss_value, "mean_q": mean_q, "mean_target": mean_target, "grad_norm": norm_value}
        if weights is not None:
            result["td_abs"] = np.asarray(values[4:], dtype=np.float32)
        return result

    # ---------------------------------------------------------------- I/O
    def save(self, path: str | Path, include_optimizer: bool = True) -> Path:
        """Write a checkpoint to exactly ``path`` (parent directories are created).

        The file holds plain tensors and primitives only, so it loads with
        ``torch.load(path, weights_only=True)``: ``q_net`` / ``target_net``
        (state dicts), ``optimizer`` (state dict, only when
        ``include_optimizer``), ``grad_steps``, ``obs_shape``, ``n_actions``
        and ``config`` (the ``DQNConfig`` fields). Tensors are saved as CPU
        tensors, so a checkpoint written on the GPU opens on any machine.
        """
        payload: dict[str, Any] = {
            "q_net": _to_cpu(self.q_net.state_dict()),
            "target_net": _to_cpu(self.target_net.state_dict()),
            "grad_steps": self.grad_steps,
            "obs_shape": list(self.obs_shape),
            "n_actions": self.n_actions,
            "config": {
                **asdict(self.cfg),
                "hidden": list(self.cfg.hidden),
                "conv_channels": list(self.cfg.conv_channels),
            },
        }
        if include_optimizer:
            payload["optimizer"] = _to_cpu(self.optimizer.state_dict())
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:  # a file object: torch.save writes exactly here
            torch.save(payload, f)
        return path

    @classmethod
    def load(cls, path: str | Path, device: str | None = None) -> "DQNAgent":
        """Inverse of :meth:`save`: networks, ``grad_steps`` and ``cfg`` round-trip exactly.

        ``device`` (``"auto"`` / ``"cpu"`` / ``"cuda"``) replaces ``cfg.device``
        of the checkpoint, e.g. to open a GPU run on a CPU-only machine. A
        checkpoint saved without the optimiser gets a fresh one.
        """
        payload = torch.load(Path(path), map_location="cpu", weights_only=True)
        # A field added to DQNConfig after the checkpoint was written takes its
        # default (an M5 checkpoint has no ``replay`` and no ``conv_channels``).
        known = {f.name for f in fields(DQNConfig)}
        cfg = DQNConfig(**{key: value for key, value in payload["config"].items() if key in known})
        if device is not None:
            cfg = replace(cfg, device=device)
        agent = cls(tuple(payload["obs_shape"]), payload["n_actions"], cfg)
        agent.q_net.load_state_dict(payload["q_net"])
        agent.target_net.load_state_dict(payload["target_net"])
        agent.grad_steps = int(payload["grad_steps"])
        if "optimizer" in payload:
            state = payload["optimizer"]
            # The fused implementation is a property of the device in use now,
            # not of the device the checkpoint was written on.
            for group in state["param_groups"]:
                group["fused"] = agent.device.type == "cuda"
            agent.optimizer.load_state_dict(state)
        return agent

    def _shape_text(self) -> str:
        return ", ".join(str(d) for d in self.obs_shape)


def _to_cpu(obj: Any) -> Any:
    """Copy of a (nested) state dict with every tensor detached on the CPU."""
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().clone()
    if isinstance(obj, dict):
        return {key: _to_cpu(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(_to_cpu(value) for value in obj)
    return obj

### 4. Experiment Configuration, Evaluation Protocol and Training Loop

#### 4.1 Experiment configuration

One YAML file (`configs/*.yaml`) becomes validated, frozen dataclasses (`EnvConfig`, `ObsConfig`, `RewardConfig`,
`TabularQConfig` or `DQNConfig`, `TrainConfig`); `--set key=value` overrides change single values, and one experiment
name is one configuration. The training environment of run seed `s` is seeded `100000 * (s + 1)`, so the generators
of different runs never overlap.

In [ ]:
# Experiment configuration: YAML file -> validated frozen dataclasses, --set overrides, seed layout
from __future__ import annotations

#: Distance between the training seed blocks of two consecutive run seeds, and
#: therefore the largest ``n_envs`` (see the module docstring).
ENV_SEED_SPACING = 100_000
MAX_N_ENVS = ENV_SEED_SPACING

ACTION_MODES: tuple[str, ...] = get_args(ActionMode)
OBS_KINDS: tuple[str, ...] = ("A", "B")
REWARD_VARIANTS: tuple[str, ...] = get_args(RewardVariant)
DEVICES: tuple[str, ...] = ("auto", "cpu", "cuda")


@dataclass(frozen=True)
class TrainConfig:
    """Training budget and evaluation protocol, shared by every agent kind."""

    #: Parallel boards of the training ``VecSnakeEnv``. (50, not a power of
    #: two, so that the round evaluation interval below is a multiple of it.)
    n_envs: int = 50
    #: Environment steps in total (``n_envs`` per vec-step).
    total_env_steps: int = 1_000_000
    #: Environment steps between two evaluations.
    eval_every: int = 50_000
    #: Greedy episodes per evaluation.
    eval_episodes: int = 100
    #: Evaluation episode ``i`` plays the board of seed ``eval_seed + i``
    #: (``evaluate.DEFAULT_EVAL_SEED``): the same boards for every agent.
    eval_seed: int = 10_000


@dataclass(frozen=True)
class ExperimentConfig:
    """One experiment: what is trained, on which environment, for how long.
    Together with a run seed it determines a run completely."""

    #: Experiment name = directory under the runs dir.
    exp: str
    env: EnvConfig
    #: ``"tabular_q"`` or ``"dqn"``.
    agent_kind: str
    #: Config dataclass of that agent kind.
    agent: TabularQConfig | DQNConfig
    train: TrainConfig


# ------------------------------------------------------------------ seeds
def env_seed_for(run_seed: int, n_envs: int) -> int:
    """Base seed of the training environment of run ``run_seed``:
    ``100000 * (run_seed + 1)``. See the module docstring for the spacing."""
    if not _is_int(run_seed) or run_seed < 0:
        raise ValueError(f"the run seed must be an integer >= 0, got {run_seed!r}")
    if not _is_int(n_envs) or not 1 <= n_envs <= MAX_N_ENVS:
        raise ValueError(f"n_envs must be an integer in 1..{MAX_N_ENVS} (the seed spacing between runs), got {n_envs!r}")
    return ENV_SEED_SPACING * (run_seed + 1)


# ------------------------------------------------------------ dict <-> config
def config_to_dict(cfg: ExperimentConfig) -> dict[str, Any]:
    """Nested plain dict in the YAML layout, every field included.

    ``config_from_dict(config_to_dict(cfg)) == cfg`` for every valid config.
    A tuple field (``hidden`` of ``DQNConfig``) is written as a list, the form
    YAML and JSON give back.
    """
    agent = {name: list(value) if isinstance(value, tuple) else value for name, value in asdict(cfg.agent).items()}
    return {
        "exp": cfg.exp,
        "env": asdict(cfg.env),
        "agent": {"kind": cfg.agent_kind, **agent},
        "train": asdict(cfg.train),
    }


def config_from_dict(raw: Mapping[str, Any]) -> ExperimentConfig:
    """Validate a nested dict in the YAML layout and build the config."""
    top = _Section(raw, "", ("exp", "env", "agent", "train", "run"))
    exp = _exp_name(top.raw.get("exp"))
    env = _env_config(top.section("env", _field_names(EnvConfig)))

    agent_raw = top.raw.get("agent")
    kind = agent_raw.get("kind") if isinstance(agent_raw, Mapping) else None
    if agent_raw is not None and not isinstance(agent_raw, Mapping):
        raise ValueError(f"agent must be a mapping, got {agent_raw!r}")
    if kind is None:
        raise ValueError(f"agent.kind is required (one of: {', '.join(_AGENT_KINDS)})")
    if not isinstance(kind, str) or kind not in _AGENT_KINDS:
        raise ValueError(f"agent.kind must be one of: {', '.join(_AGENT_KINDS)}; got {kind!r}")
    config_type, parse_agent = _AGENT_KINDS[kind]
    try:
        agent_section = top.section("agent", ("kind", *_field_names(config_type)))
    except ValueError as err:  # an unknown key: the keys that exist depend on the kind, so say which one it is
        raise ValueError(f"{err} for agent.kind {kind}") from None
    agent = parse_agent(agent_section)

    train = _train_config(top.section("train", _field_names(TrainConfig)))
    cfg = ExperimentConfig(exp=exp, env=env, agent_kind=kind, agent=agent, train=train)
    _check_combination(cfg)
    return cfg


class _UniqueKeyLoader(yaml.SafeLoader):
    """``yaml.safe_load`` that refuses a repeated key. PyYAML keeps the last
    one silently, so a second ``train:`` block would drop the keys of the
    first one back to their defaults."""

    def construct_mapping(self, node: yaml.MappingNode, deep: bool = False) -> dict[Any, Any]:
        seen: set[Any] = set()
        for key_node, _ in node.value:
            key = self.construct_object(key_node, deep=True)
            if key in seen:
                raise yaml.constructor.ConstructorError(None, None, f"duplicate key {key!r}", key_node.start_mark)
            seen.add(key)
        return super().construct_mapping(node, deep)


def load_config(path: str | Path, overrides: Sequence[str] = ()) -> ExperimentConfig:
    """Read a YAML experiment file, apply ``dotted.key=value`` overrides in
    order, validate. ``ValueError`` (prefixed with the path) on any problem."""
    path = Path(path)
    if isinstance(overrides, str):
        overrides = (overrides,)
    try:
        raw = yaml.load(path.read_text(encoding="utf-8"), Loader=_UniqueKeyLoader)
        if raw is None:
            raw = {}
        if not isinstance(raw, dict):
            raise ValueError(f"the top level must be a mapping, got {raw!r}")
        for item in overrides:
            _apply_override(raw, item)
        return config_from_dict(raw)
    except yaml.YAMLError as err:
        raise ValueError(f"{path}: not valid YAML: {err}") from err
    except ValueError as err:
        raise ValueError(f"{path}: {err}") from err


def _apply_override(raw: dict[str, Any], item: str) -> None:
    """Set ``dotted.key=value`` in the nested dict; the value is parsed as YAML."""
    key, sep, text = str(item).partition("=")
    parts = key.strip().split(".")
    if not sep or not all(parts):
        raise ValueError(f"override {item!r} is not of the form dotted.key=value")
    try:
        value = yaml.safe_load(text)
    except yaml.YAMLError as err:
        raise ValueError(f"override {item!r}: the value is not valid YAML: {err}") from err
    node = raw
    for depth, part in enumerate(parts[:-1], start=1):
        child = node.get(part)
        if child is None:
            child = node[part] = {}
        elif not isinstance(child, dict):
            raise ValueError(f"override {item!r}: {'.'.join(parts[:depth])} is not a section")
        node = child
    node[parts[-1]] = value


# ----------------------------------------------------------------- parsing
def _is_int(value: object) -> bool:
    """A real integer: ``True`` is not one, and neither is ``128.0``."""
    return isinstance(value, int) and not isinstance(value, bool)


def _field_names(config_type: type) -> tuple[str, ...]:
    return tuple(f.name for f in fields(config_type))


def _require(ok: bool, message: str) -> None:
    if not ok:
        raise ValueError(message)


class _Section:
    """One mapping of the configuration with its dotted path. The typed
    getters return the value of a key, or ``default`` when the key is absent,
    and name the key in every error."""

    def __init__(self, raw: object, path: str, allowed: Sequence[str]) -> None:
        if raw is None:  # "env:" with every line commented out
            raw = {}
        if not isinstance(raw, Mapping):
            raise ValueError(f"{path or 'the top level'} must be a mapping, got {raw!r}")
        self.raw = raw
        self.path = path
        for name in raw:
            if name not in allowed:
                raise ValueError(f"unknown key {self.key(name)} (known keys here: {', '.join(allowed)})")

    def key(self, name: object) -> str:
        return f"{self.path}.{name}" if self.path else str(name)

    def section(self, name: str, allowed: Sequence[str]) -> "_Section":
        return _Section(self.raw.get(name), self.key(name), allowed)

    def integer(self, name: str, default: int) -> int:
        value = self.raw.get(name, default)
        if not _is_int(value):
            raise ValueError(f"{self.key(name)} must be an integer, got {value!r}")
        return value

    def optional_integer(self, name: str, default: int | None) -> int | None:
        """An integer or null (``None``)."""
        value = self.raw.get(name, default)
        if value is not None and not _is_int(value):
            raise ValueError(f"{self.key(name)} must be null or an integer, got {value!r}")
        return value

    def number(self, name: str, default: float) -> float:
        value = self.raw.get(name, default)
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            hint = ""
            # PyYAML reads YAML 1.1: a float needs a dot in the mantissa AND a
            # sign in the exponent; 1e-3 and 1.0e9 are both strings to it.
            if isinstance(value, str) and re.fullmatch(r"[+-]?\d+(\.\d*)?[eE][+-]?\d+", value.strip()):
                hint = " (YAML needs a dot in the mantissa and a sign in the exponent: write 1.0e-3 or 1.0e+9)"
            raise ValueError(f"{self.key(name)} must be a finite number, got {value!r}{hint}")
        return float(value)

    def boolean(self, name: str, default: bool) -> bool:
        value = self.raw.get(name, default)
        if not isinstance(value, bool):
            raise ValueError(f"{self.key(name)} must be true or false, got {value!r}")
        return value

    def choice(self, name: str, default: str, allowed: Sequence[str]) -> str:
        value = self.raw.get(name, default)
        if not isinstance(value, str) or value not in allowed:
            raise ValueError(f"{self.key(name)} must be one of: {', '.join(allowed)}; got {value!r}")
        return value


_EXP_NAME = re.compile(r"[A-Za-z0-9._=+-]+")
#: Windows refuses these as file or directory names, with or without a suffix.
_WINDOWS_DEVICE_NAMES = frozenset(
    ["CON", "PRN", "AUX", "NUL", *(f"COM{i}" for i in range(1, 10)), *(f"LPT{i}" for i in range(1, 10))]
)


def _exp_name(value: object) -> str:
    if value is None:
        raise ValueError("exp is required: the experiment name, used as the directory under the runs dir")
    if not isinstance(value, str):
        raise ValueError(f"exp must be a string, got {value!r} (quote it)")
    usable = (
        _EXP_NAME.fullmatch(value) is not None
        and not value.endswith(".")  # ".", ".." and names Windows would silently shorten
        and value.split(".")[0].upper() not in _WINDOWS_DEVICE_NAMES
    )
    _require(
        usable,
        f"exp must be a non-empty directory name made of letters, digits and . _ - = + only "
        f"(not ending in a dot, not a Windows device name), got {value!r}",
    )
    return value


def _env_config(s: _Section) -> EnvConfig:
    default = EnvConfig()
    W = s.integer("W", default.W)
    _require(W >= 4, f"{s.key('W')} must be at least 4, got {W}")
    starve_limit = s.raw.get("starve_limit", default.starve_limit)
    if starve_limit is not None:
        starve_limit = s.integer("starve_limit", 0)
        _require(starve_limit >= 1, f"{s.key('starve_limit')} must be null or at least 1, got {starve_limit}")
    action_mode = s.choice("action_mode", default.action_mode, ACTION_MODES)

    o = s.section("obs", _field_names(ObsConfig))
    obs = ObsConfig(
        kind=o.choice("kind", default.obs.kind, OBS_KINDS),
        age_plane=o.boolean("age_plane", default.obs.age_plane),
        heading_planes=o.boolean("heading_planes", default.obs.heading_planes),
    )
    r = s.section("reward", _field_names(RewardConfig))
    numbers = [name for name in _field_names(RewardConfig) if name != "variant"]
    reward = RewardConfig(
        variant=r.choice("variant", default.reward.variant, REWARD_VARIANTS),
        **{name: r.number(name, getattr(default.reward, name)) for name in numbers},
    )
    return EnvConfig(W=W, action_mode=action_mode, starve_limit=starve_limit, obs=obs, reward=reward)


def _tabular_q_config(s: _Section) -> TabularQConfig:
    default = TabularQConfig()
    cfg = TabularQConfig(**{name: s.number(name, getattr(default, name)) for name in _field_names(TabularQConfig)})
    _require(0 < cfg.alpha <= 1, f"{s.key('alpha')} must be in (0, 1], got {cfg.alpha}")
    _require(0 <= cfg.gamma <= 1, f"{s.key('gamma')} must be in [0, 1], got {cfg.gamma}")
    _require(
        0 <= cfg.eps_end <= cfg.eps_start <= 1,
        f"need 0 <= {s.key('eps_end')} <= {s.key('eps_start')} <= 1, "
        f"got eps_end {cfg.eps_end} and eps_start {cfg.eps_start}",
    )
    _require(
        0 <= cfg.eps_decay_fraction <= 1,
        f"{s.key('eps_decay_fraction')} must be in [0, 1], got {cfg.eps_decay_fraction}",
    )
    return cfg


def _train_config(s: _Section) -> TrainConfig:
    default = TrainConfig()
    cfg = TrainConfig(**{name: s.integer(name, getattr(default, name)) for name in _field_names(TrainConfig)})
    n_envs, every, total = cfg.n_envs, cfg.eval_every, cfg.total_env_steps
    _require(1 <= n_envs <= MAX_N_ENVS, f"{s.key('n_envs')} must be in 1..{MAX_N_ENVS}, got {n_envs}")
    _require(
        every >= n_envs and every % n_envs == 0,
        f"{s.key('eval_every')} ({every}) must be a positive multiple of {s.key('n_envs')} ({n_envs}), "
        f"so that evaluations land on exact step counts",
    )
    _require(
        total >= every and total % every == 0,
        f"{s.key('total_env_steps')} ({total}) must be a positive multiple of {s.key('eval_every')} ({every})",
    )
    _require(cfg.eval_episodes >= 2, f"{s.key('eval_episodes')} must be at least 2, got {cfg.eval_episodes}")
    _require(
        cfg.eval_seed >= 0 and cfg.eval_seed + cfg.eval_episodes <= ENV_SEED_SPACING,
        f"{s.key('eval_seed')} ({cfg.eval_seed}): the evaluation boards eval_seed .. eval_seed + eval_episodes - 1 "
        f"must lie in 0..{ENV_SEED_SPACING - 1}, below the first training seed block",
    )
    return cfg


def _dqn_config(s: _Section) -> DQNConfig:
    default = DQNConfig()
    hidden = s.raw.get("hidden", list(default.hidden))
    _require(
        isinstance(hidden, (list, tuple)) and all(_is_int(width) and width >= 1 for width in hidden),
        f"{s.key('hidden')} must be a list of positive integers (possibly empty: a linear Q-function), got {hidden!r}",
    )
    conv_channels = s.raw.get("conv_channels", list(default.conv_channels))
    _require(
        isinstance(conv_channels, (list, tuple)) and all(_is_int(c) and c >= 1 for c in conv_channels),
        f"{s.key('conv_channels')} must be a list of positive integers (possibly empty: no convolution), "
        f"got {conv_channels!r}",
    )
    cfg = DQNConfig(
        gamma=s.number("gamma", default.gamma),
        lr=s.number("lr", default.lr),
        batch_size=s.integer("batch_size", default.batch_size),
        buffer_size=s.integer("buffer_size", default.buffer_size),
        replay=s.boolean("replay", default.replay),
        learning_starts=s.integer("learning_starts", default.learning_starts),
        grad_steps_per_vec_step=s.optional_integer("grad_steps_per_vec_step", default.grad_steps_per_vec_step),
        target_update_every=s.integer("target_update_every", default.target_update_every),
        grad_clip=s.number("grad_clip", default.grad_clip),
        eps_start=s.number("eps_start", default.eps_start),
        eps_end=s.number("eps_end", default.eps_end),
        eps_decay_fraction=s.number("eps_decay_fraction", default.eps_decay_fraction),
        hidden=tuple(hidden),
        conv_channels=tuple(conv_channels),
        double=s.boolean("double", default.double),
        dueling=s.boolean("dueling", default.dueling),
        per=s.boolean("per", default.per),
        per_alpha=s.number("per_alpha", default.per_alpha),
        per_beta_start=s.number("per_beta_start", default.per_beta_start),
        per_eps=s.number("per_eps", default.per_eps),
        keep_checkpoints=s.boolean("keep_checkpoints", default.keep_checkpoints),
        device=s.choice("device", default.device, DEVICES),
        torch_threads=s.optional_integer("torch_threads", default.torch_threads),
    )
    _require(0 <= cfg.gamma <= 1, f"{s.key('gamma')} must be in [0, 1], got {cfg.gamma}")
    _require(cfg.lr > 0, f"{s.key('lr')} must be positive, got {cfg.lr}")
    _require(cfg.batch_size >= 1, f"{s.key('batch_size')} must be at least 1, got {cfg.batch_size}")
    _require(
        cfg.buffer_size >= cfg.batch_size,
        f"{s.key('buffer_size')} ({cfg.buffer_size}) must be at least {s.key('batch_size')} ({cfg.batch_size})",
    )
    _require(
        cfg.learning_starts >= cfg.batch_size,
        f"{s.key('learning_starts')} ({cfg.learning_starts}) must be at least {s.key('batch_size')} "
        f"({cfg.batch_size}): the first gradient step needs a batch of transitions",
    )
    _require(
        cfg.grad_steps_per_vec_step is None or cfg.grad_steps_per_vec_step >= 1,
        f"{s.key('grad_steps_per_vec_step')} must be null or at least 1, got {cfg.grad_steps_per_vec_step}",
    )
    _require(
        cfg.replay or cfg.grad_steps_per_vec_step in (None, 1),
        f"{s.key('grad_steps_per_vec_step')} must be null or 1 when {s.key('replay')} is false: without replay "
        f"there is exactly one gradient step per vec-step, on that step's own transitions; "
        f"got {cfg.grad_steps_per_vec_step}",
    )
    _require(
        cfg.target_update_every >= 1,
        f"{s.key('target_update_every')} must be at least 1, got {cfg.target_update_every}",
    )
    _require(cfg.grad_clip > 0, f"{s.key('grad_clip')} must be positive, got {cfg.grad_clip}")
    _require(
        0 <= cfg.eps_end <= cfg.eps_start <= 1,
        f"need 0 <= {s.key('eps_end')} <= {s.key('eps_start')} <= 1, "
        f"got eps_end {cfg.eps_end} and eps_start {cfg.eps_start}",
    )
    _require(
        0 <= cfg.eps_decay_fraction <= 1,
        f"{s.key('eps_decay_fraction')} must be in [0, 1], got {cfg.eps_decay_fraction}",
    )
    _require(
        cfg.torch_threads is None or cfg.torch_threads >= 1,
        f"{s.key('torch_threads')} must be null or at least 1, got {cfg.torch_threads}",
    )
    _require(0 <= cfg.per_alpha <= 1, f"{s.key('per_alpha')} must be in [0, 1], got {cfg.per_alpha}")
    _require(
        0 <= cfg.per_beta_start <= 1, f"{s.key('per_beta_start')} must be in [0, 1], got {cfg.per_beta_start}"
    )
    _require(cfg.per_eps > 0, f"{s.key('per_eps')} must be positive, got {cfg.per_eps}")
    _require(
        cfg.replay or not cfg.per,
        f"{s.key('per')} needs {s.key('replay')}: prioritised replay is a way of sampling the replay buffer",
    )
    return cfg


#: ``agent.kind`` -> (config dataclass, parser of the ``agent`` section).
_AGENT_KINDS: dict[str, tuple[type, Callable[[_Section], Any]]] = {
    "tabular_q": (TabularQConfig, _tabular_q_config),
    "dqn": (DQNConfig, _dqn_config),
}


def _check_combination(cfg: ExperimentConfig) -> None:
    """Rules that tie two sections together."""
    env, agent = cfg.env, cfg.agent
    if cfg.agent_kind == "tabular_q":
        _require(
            env.obs.kind == "A",
            f"env.obs.kind must be A for agent.kind tabular_q (the table is indexed by the 11 binary "
            f"features), got {env.obs.kind!r}",
        )
    if cfg.agent_kind == "dqn":
        _require(
            agent.learning_starts <= cfg.train.total_env_steps,
            f"agent.learning_starts ({agent.learning_starts}) must not exceed train.total_env_steps "
            f"({cfg.train.total_env_steps}): the run would end before the first gradient step",
        )
        _require(
            not agent.replay or agent.buffer_size >= cfg.train.n_envs,
            f"agent.buffer_size ({agent.buffer_size}) must be at least train.n_envs ({cfg.train.n_envs}) with "
            f"replay: a smaller buffer would drop part of every vec-step's transitions",
        )
    if env.reward.variant == "R2":
        _require(
            env.reward.gamma == agent.gamma,
            f"env.reward.gamma ({env.reward.gamma}) must equal agent.gamma ({agent.gamma}) under reward R2: "
            f"potential-based shaping keeps the optimal policy only for the agent's discount; set both",
        )

#### 4.2 Evaluation protocol

`evaluate(policy, cfg, n_episodes, seed)` plays greedy episodes on the scalar environment, episode `i` reset with
`seed + i`: the same boards for every agent and every reference player. Per episode: score, steps, length and the end
cause; the summary gives the mean score with its 95 % interval, the mean steps and the cause breakdown.
`evaluate_batched` is the same protocol on the vectorised environment (the first episode of each sub-environment),
used by the training loop; with `gamma` both also record the value estimate `max_a Q` against the realised discounted
return, the overestimation measurement of the Double DQN study. Results serialise to JSON with the environment
configuration and the git hash.

In [ ]:
# Evaluation protocol: greedy episodes on fixed boards, death causes, value estimate against realised return
from __future__ import annotations

DEFAULT_EVAL_SEED = 10_000
#: The per-episode value fields (M7), present only when evaluated with ``gamma``.
VALUE_FIELDS = ("value_start", "return_start", "value_visited", "return_visited")


@dataclass(frozen=True)
class EpisodeRecord:
    seed: int
    score: int
    steps: int
    length: int
    cause: str
    #: ``max_a Q(s_0, a)`` of the first state (M7; ``None`` without ``gamma``).
    value_start: float | None = None
    #: Realised discounted return from the first state.
    return_start: float | None = None
    #: Mean of ``max_a Q(s_t, a)`` over the episode's steps.
    value_visited: float | None = None
    #: Mean of the realised discounted return ``G_t`` over the episode's steps.
    return_visited: float | None = None


@dataclass
class EvalResult:
    policy: str
    env: dict[str, Any]
    seed: int
    episodes: list[EpisodeRecord] = field(default_factory=list)
    git: str = "unknown"
    wall_time_s: float = 0.0

    # ----------------------------------------------------------- summaries
    @property
    def scores(self) -> np.ndarray:
        return np.array([e.score for e in self.episodes], dtype=np.float64)

    @property
    def steps(self) -> np.ndarray:
        return np.array([e.steps for e in self.episodes], dtype=np.float64)

    @property
    def mean_score(self) -> float:
        return float(self.scores.mean())

    @property
    def ci95(self) -> float:
        """Half-width of the 95 % CI of the mean score (normal approximation)."""
        n = len(self.episodes)
        if n < 2:
            return float("nan")
        return float(1.96 * self.scores.std(ddof=1) / np.sqrt(n))

    @property
    def cause_fractions(self) -> dict[str, float]:
        n = max(len(self.episodes), 1)
        return {c: sum(e.cause == c for e in self.episodes) / n for c in CAUSE_NAMES if c != "none"}

    def summary(self) -> dict[str, Any]:
        return {
            "policy": self.policy,
            "n_episodes": len(self.episodes),
            "mean_score": self.mean_score,
            "ci95": self.ci95,
            "median_score": float(np.median(self.scores)) if self.episodes else float("nan"),
            "max_score": int(self.scores.max()) if self.episodes else 0,
            "mean_steps": float(self.steps.mean()) if self.episodes else float("nan"),
            "cause_fractions": self.cause_fractions,
        }

    def value_summary(self) -> dict[str, float] | None:
        """The value estimates against the realised returns (module docstring).

        ``value_start`` / ``return_start``: means over the episodes of the
        first-state quantities. ``value_visited`` / ``return_visited``: means
        over all visited states (the per-episode means weighted by the
        episode's steps). ``None`` when there are no episodes or an episode
        was evaluated without ``gamma``.
        """
        if not self.episodes or any(getattr(e, name) is None for e in self.episodes for name in VALUE_FIELDS):
            return None
        steps = self.steps
        total_steps = float(steps.sum())
        return {
            "value_start": float(np.mean([e.value_start for e in self.episodes])),
            "return_start": float(np.mean([e.return_start for e in self.episodes])),
            "value_visited": float(np.dot([e.value_visited for e in self.episodes], steps) / total_steps),
            "return_visited": float(np.dot([e.return_visited for e in self.episodes], steps) / total_steps),
        }

    # --------------------------------------------------------------- I/O
    def to_dict(self) -> dict[str, Any]:
        d = {
            "policy": self.policy,
            "env": self.env,
            "seed": self.seed,
            "git": self.git,
            "wall_time_s": self.wall_time_s,
            "summary": self.summary(),
        }
        values = self.value_summary()
        if values is not None:
            d["values"] = values
        d["episodes"] = [_episode_dict(e) for e in self.episodes]
        return d

    def save(self, path: str | Path) -> Path:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(self.to_dict(), indent=1), encoding="utf-8")
        return path

    @classmethod
    def load(cls, path: str | Path) -> "EvalResult":
        d = json.loads(Path(path).read_text(encoding="utf-8"))
        return cls(
            policy=d["policy"],
            env=d["env"],
            seed=d["seed"],
            episodes=[EpisodeRecord(**e) for e in d["episodes"]],
            git=d.get("git", "unknown"),
            wall_time_s=d.get("wall_time_s", 0.0),
        )


def _episode_dict(e: EpisodeRecord) -> dict[str, Any]:
    """The JSON form of a record: the value fields only when present, so a
    file written without ``gamma`` is what it was before M7."""
    d = asdict(e)
    for name in VALUE_FIELDS:
        if d[name] is None:
            del d[name]
    return d


def env_config_to_dict(cfg: EnvConfig) -> dict[str, Any]:
    return asdict(cfg)


def env_config_from_dict(d: dict[str, Any]) -> EnvConfig:
    return EnvConfig(
        W=d["W"],
        action_mode=d["action_mode"],
        starve_limit=d["starve_limit"],
        obs=ObsConfig(**d["obs"]),
        reward=RewardConfig(**d["reward"]),
    )


# ------------------------------------------------------- value bookkeeping
def _check_gamma(policy, gamma: float | None) -> None:
    if gamma is None:
        return
    if isinstance(gamma, bool) or not isinstance(gamma, (int, float, np.floating)) or not 0.0 <= gamma <= 1.0:
        raise ValueError(f"gamma must be None or a number in [0, 1], got {gamma!r}")
    if not callable(getattr(policy, "q_values", None)):
        raise TypeError(
            f"value estimates need a policy with q_values(obs) -> (N, n_actions); {type(policy).__name__} has none"
        )


def _value_fields(values: list[float], rewards: list[float], gamma: float) -> dict[str, float]:
    """The four per-episode value fields from the per-step ``max_a Q`` and rewards."""
    returns = np.empty(len(rewards), dtype=np.float64)
    g = 0.0
    for t in range(len(rewards) - 1, -1, -1):  # G_t = r_t + gamma * G_{t+1}, G_T = 0
        g = rewards[t] + gamma * g
        returns[t] = g
    return {
        "value_start": float(values[0]),
        "return_start": float(returns[0]),
        "value_visited": float(np.mean(values)),
        "return_visited": float(returns.mean()),
    }


def evaluate(
    policy, cfg: EnvConfig, n_episodes: int = 100, seed: int = DEFAULT_EVAL_SEED, gamma: float | None = None
) -> EvalResult:
    """Play ``n_episodes`` greedy episodes; episode ``i`` uses ``reset(seed=seed + i)``.

    With ``gamma`` the records also carry the value estimates against the
    realised returns (module docstring); the policy must have ``q_values``.
    """
    _check_gamma(policy, gamma)
    env = SnakeEnv(cfg)
    result = EvalResult(policy=policy.name, env=env_config_to_dict(cfg), seed=seed, git=git_hash())
    t0 = time.perf_counter()
    for i in range(n_episodes):
        ep_seed = seed + i
        obs, info = env.reset(seed=ep_seed)
        if hasattr(policy, "reset"):
            policy.reset()
        values: list[float] = []
        rewards: list[float] = []
        while True:
            if gamma is not None:
                values.append(float(policy.q_values(obs[None])[0].max()))
            obs, reward, terminated, truncated, info = env.step(policy.act(obs, env))
            if gamma is not None:
                rewards.append(float(reward))
            if terminated or truncated:
                break
        extra = _value_fields(values, rewards, gamma) if gamma is not None else {}
        result.episodes.append(
            EpisodeRecord(
                seed=ep_seed, score=info["score"], steps=info["steps"], length=info["length"], cause=info["cause"],
                **extra,
            )
        )
    result.wall_time_s = time.perf_counter() - t0
    return result


def evaluate_batched(
    policy, cfg: EnvConfig, n_episodes: int = 100, seed: int = DEFAULT_EVAL_SEED, gamma: float | None = None
) -> EvalResult:
    """Greedy evaluation on the vectorised env: the first episode of each of
    ``n_episodes`` sub-envs, sub-env ``i`` seeded ``seed + i``.

    ``policy`` needs ``name`` and ``act_batch(obs (N, *obs_shape)) -> (N,)``
    integer actions and must be a deterministic function of the observation;
    the result then equals ``evaluate(policy, cfg, n_episodes, seed)`` episode
    by episode (see the module docstring). Episodes are ordered by ``i``.
    ``policy.reset()``, if it exists, is called once before the first step
    (all first episodes start together). With ``gamma`` (module docstring)
    the policy must also have ``q_values``; it is called once per vec-step on
    the whole batch, before ``act_batch``.
    """
    if n_episodes < 1:
        raise ValueError(f"n_episodes must be at least 1, got {n_episodes}")
    _check_gamma(policy, gamma)
    env = VecSnakeEnv(cfg, n_envs=n_episodes, seed=seed)
    result = EvalResult(policy=policy.name, env=env_config_to_dict(cfg), seed=seed, git=git_hash())
    t0 = time.perf_counter()
    if hasattr(policy, "reset"):
        policy.reset()
    obs = env.reset()
    # ``info`` reports the counters of the episode that just ended (the state
    # arrays of the env already hold the next one), so the record of a sub-env
    # is copied from ``info`` on the step that ends its first episode.
    keys = ("score", "steps", "length", "cause")
    first = {key: np.zeros(n_episodes, dtype=np.int64) for key in keys}
    pending = np.ones(n_episodes, dtype=bool)  # first episode still running
    step_values: list[np.ndarray] = []  # per vec-step: max_a Q of every sub-env (with gamma)
    step_rewards: list[np.ndarray] = []
    while pending.any():
        if gamma is not None:
            step_values.append(policy.q_values(obs).max(axis=1))
        obs, reward, _, _, info = env.step(policy.act_batch(obs))
        if gamma is not None:
            step_rewards.append(np.asarray(reward, dtype=np.float64))
        ended = pending & info["done"]
        for key in keys:
            first[key][ended] = info[key][ended]
        pending &= ~ended
    columns = [first[key].tolist() for key in keys]  # Python ints: JSON-serialisable
    if gamma is not None:
        all_values = np.stack(step_values)  # (vec-steps, n_episodes)
        all_rewards = np.stack(step_rewards)
    for i, (score, steps, length, cause) in enumerate(zip(*columns)):
        extra = {}
        if gamma is not None:
            # The first episode of sub-env i occupies its first ``steps`` vec-steps.
            extra = _value_fields(all_values[:steps, i].tolist(), all_rewards[:steps, i].tolist(), gamma)
        result.episodes.append(
            EpisodeRecord(seed=seed + i, score=score, steps=steps, length=length, cause=CAUSE_NAMES[cause], **extra)
        )
    result.wall_time_s = time.perf_counter() - t0
    return result


def markdown_table(results: list[EvalResult]) -> str:
    causes = [c for c in CAUSE_NAMES if c != "none"]
    head = "| policy | episodes | score mean +- 95% CI | median | max | mean steps | " + " | ".join(causes) + " |"
    sep = "|" + "---|" * (6 + len(causes))
    rows = [head, sep]
    for r in results:
        s = r.summary()
        frac = " | ".join(f"{s['cause_fractions'][c]:.2f}" for c in causes)
        rows.append(
            f"| {s['policy']} | {s['n_episodes']} | {s['mean_score']:.2f} +- {s['ci95']:.2f} | "
            f"{s['median_score']:.1f} | {s['max_score']} | {s['mean_steps']:.1f} | {frac} |"
        )
    return "\n".join(rows)

#### 4.3 Training

`train(cfg, seed, runs_dir)` writes `<runs_dir>/<exp>/<seed>/`: `config.yaml` (every field, the git hash and the
software versions, written before training starts), `metrics.csv` (one row per evaluation), `q_table.npz` or
`checkpoints/` and `model.pt`, and `final_eval.json`, whose presence marks a finished run; a run is never overwritten.
The tabular loop: epsilon from the linear schedule, `act_epsilon` on `N` boards, one `step` of the vectorised
environment, `update` on the true successors `info["final_obs"]` with the `terminated` flags, an evaluation on the
fixed boards every `eval_every` steps. The DQN loop puts the replay buffer between acting and learning: each vec-step
adds `N` transitions and, after `learning_starts`, is followed by `G` gradient steps on sampled batches (default one
gradient step per four environment steps); without replay (`agent.replay: false`) exactly one gradient step on the
vec-step's own transitions. A non-finite loss aborts the run (`aborted.json`). The command-line wrapper `main` is
left out; it parses `--config --seed --exp --set --runs-dir` and calls `train`.

In [ ]:
# Config-driven training: the tabular and the DQN trainers, the run directory and its files
from __future__ import annotations

CONFIG_FILE = "config.yaml"
METRICS_FILE = "metrics.csv"
Q_TABLE_FILE = "q_table.npz"
MODEL_FILE = "model.pt"
CHECKPOINT_DIR = "checkpoints"
FINAL_EVAL_FILE = "final_eval.json"
ABORTED_FILE = "aborted.json"
#: Everything a run writes, of either agent kind (``q_table.npz``: tabular_q;
#: ``model.pt``, ``checkpoints/`` and ``aborted.json``: dqn). A directory
#: holding any of them is a run.
RUN_FILES = (CONFIG_FILE, METRICS_FILE, Q_TABLE_FILE, MODEL_FILE, CHECKPOINT_DIR, FINAL_EVAL_FILE, ABORTED_FILE)

#: Evaluation columns shared by every agent kind.
EVAL_COLUMNS = (
    "eval_mean_score",
    "eval_ci95",
    "eval_median_score",
    "eval_max_score",
    "eval_mean_steps",
    "eval_frac_wall",
    "eval_frac_self",
    "eval_frac_starve",
    "eval_frac_win",
)
#: Header of ``metrics.csv`` for tabular Q-learning, in file order.
TABULAR_COLUMNS = (
    "env_steps",
    "vec_steps",
    "epsilon",
    *EVAL_COLUMNS,
    "train_episodes",
    "train_mean_score",
    "mean_abs_td",
    "cells_visited",
    "wall_time_s",
)
#: The value estimates against the realised returns of an evaluation (M7;
#: ``EvalResult.value_summary()``), DQN only.
VALUE_COLUMNS = ("eval_value_start", "eval_return_start", "eval_value_visited", "eval_return_visited")
#: The statistics ``DQNAgent.update`` returns, averaged per metrics row.
UPDATE_STATS = ("loss", "mean_q", "mean_target", "grad_norm")
#: The phases of the DQN loop that are timed (``time_<phase>_s``).
PHASES = ("env", "act", "update", "eval")
#: Header of ``metrics.csv`` for DQN, in file order.
DQN_COLUMNS = (
    "env_steps",
    "vec_steps",
    "grad_steps",
    "epsilon",
    *EVAL_COLUMNS,
    *VALUE_COLUMNS,
    "train_episodes",
    "train_mean_score",
    "train_truncated",
    *UPDATE_STATS,
    "buffer_size",
    *(f"time_{phase}_s" for phase in PHASES),
    "wall_time_s",
)


# ---------------------------------------------------------------- the loop
def _train_tabular_q(cfg: ExperimentConfig, seed: int, run_dir: Path, quiet: bool) -> None:
    """Tabular Q-learning on observation A (the loop of the module docstring)."""
    ac, tc = cfg.agent, cfg.train
    env = VecSnakeEnv(cfg.env, tc.n_envs, seed=env_seed_for(seed, tc.n_envs))
    agent = TabularQAgent(env.n_actions, ac)
    explore_rng = np.random.default_rng([seed, 1])  # exploration stream, apart from every food stream
    decay_steps = ac.eps_decay_fraction * tc.total_env_steps

    with MetricsWriter(run_dir / METRICS_FILE, TABULAR_COLUMNS) as metrics:
        evaluate_now = partial(
            _evaluate_tabular, agent, cfg, metrics, started=time.perf_counter(), label=f"{cfg.exp}/{seed}", quiet=quiet
        )
        obs = env.reset()
        last_eval = evaluate_now(0, ac.eps_start, TrainWindow())  # before any update
        window = TrainWindow()
        for k in range(tc.total_env_steps // tc.n_envs):
            epsilon = linear_schedule(k * tc.n_envs, ac.eps_start, ac.eps_end, decay_steps)
            actions = agent.act_epsilon(obs, epsilon, explore_rng)
            next_obs, reward, terminated, truncated, info = env.step(actions)
            # True successor (not the post-reset observation); bootstrap unless
            # terminated, so a starvation truncation keeps its future value.
            abs_td = agent.update(obs, actions, reward, info["final_obs"], terminated)
            obs = next_obs
            window.add(info, abs_td)

            env_steps = (k + 1) * tc.n_envs
            if env_steps % tc.eval_every == 0:
                last_eval = evaluate_now(env_steps, epsilon, window)
                window = TrainWindow()

    agent.save(run_dir / Q_TABLE_FILE)
    _save_final_eval(last_eval, run_dir)


@dataclass
class TrainWindow:
    """Training statistics accumulated between two metrics rows."""

    episodes: int = 0
    score_sum: int = 0
    updates: int = 0
    abs_td_sum: float = 0.0

    def add(self, info: Mapping[str, np.ndarray], abs_td: float) -> None:
        """One vec-step: the episodes it finished and the mean |TD error| of its update."""
        done = info["done"]
        self.episodes += int(done.sum())
        self.score_sum += int(info["score"][done].sum())
        self.updates += 1
        self.abs_td_sum += abs_td

    @property
    def mean_score(self) -> float | None:
        return self.score_sum / self.episodes if self.episodes else None

    @property
    def mean_abs_td(self) -> float | None:
        return self.abs_td_sum / self.updates if self.updates else None


def _evaluate_tabular(
    agent: TabularQAgent,
    cfg: ExperimentConfig,
    metrics: "MetricsWriter",
    env_steps: int,
    epsilon: float,
    window: TrainWindow,
    *,
    started: float,
    label: str,
    quiet: bool,
) -> EvalResult:
    """Greedy evaluation on the fixed evaluation boards; appends one metrics row."""
    tc = cfg.train
    result = evaluate_batched(agent, cfg.env, tc.eval_episodes, tc.eval_seed)
    row = {
        "env_steps": env_steps,
        "vec_steps": env_steps // tc.n_envs,
        "epsilon": epsilon,
        **eval_columns(result),
        "train_episodes": window.episodes,
        "train_mean_score": window.mean_score,
        "mean_abs_td": window.mean_abs_td,
        "cells_visited": int(np.count_nonzero(agent.visits)),
        "wall_time_s": round(time.perf_counter() - started, 3),
    }
    metrics.write(row)
    if not quiet:
        print(_progress_line(label, tc.total_env_steps, row), flush=True)
    return result


def _train_dqn(cfg: ExperimentConfig, seed: int, run_dir: Path, quiet: bool) -> None:
    """DQN with a target network, with uniform replay (``agent.replay``) or
    without it (the two DQN loops of the module docstring)."""
    import torch  # here, not at module level: tabular runs and --help never import torch

    ac, tc = cfg.agent, cfg.train
    if ac.torch_threads is not None:
        torch.set_num_threads(ac.torch_threads)
    env = VecSnakeEnv(cfg.env, tc.n_envs, seed=env_seed_for(seed, tc.n_envs))
    agent = DQNAgent(env.obs_shape, env.n_actions, ac, seed=seed)  # initial weights: a function of the run seed
    buffer: ReplayBuffer | None = None
    if ac.replay:
        buffer = (
            PrioritizedReplayBuffer(ac.buffer_size, env.obs_shape, ac.per_alpha, ac.per_eps)
            if ac.per
            else ReplayBuffer(ac.buffer_size, env.obs_shape)
        )
    explore_rng = np.random.default_rng([seed, 1])  # exploration stream, as in the tabular loop
    sample_rng = np.random.default_rng([seed, 2])  # replay sampling stream (created, never drawn without replay)
    grad_steps_per_vec_step = (ac.grad_steps_per_vec_step or max(1, tc.n_envs // 4)) if ac.replay else 1
    decay_steps = ac.eps_decay_fraction * tc.total_env_steps

    def learn_from_buffer(env_steps_before: int) -> None:
        """The gradient steps of one vec-step with replay: uniform, or prioritised (module docstring)."""
        if isinstance(buffer, PrioritizedReplayBuffer):
            beta = linear_schedule(env_steps_before, ac.per_beta_start, 1.0, tc.total_env_steps)
            for _ in range(grad_steps_per_vec_step):
                batch = buffer.sample(ac.batch_size, sample_rng, beta)
                stats = agent.update(batch, batch.weights)
                buffer.update_priorities(batch.indices, stats["td_abs"])
                window.add_update(stats)
        else:
            for _ in range(grad_steps_per_vec_step):
                window.add_update(agent.update(buffer.sample(ac.batch_size, sample_rng)))
    # Queued GPU work is waited for at row boundaries only, never per update.
    timer = PhaseTimer(synchronize=torch.cuda.synchronize if agent.device.type == "cuda" else None)

    env_steps = 0
    try:
        with MetricsWriter(run_dir / METRICS_FILE, DQN_COLUMNS) as metrics:
            evaluate_now = partial(
                _evaluate_dqn, agent, buffer, cfg, metrics, run_dir, timer, label=f"{cfg.exp}/{seed}", quiet=quiet
            )
            obs = env.reset()
            last_eval = evaluate_now(0, ac.eps_start, DQNWindow())  # before any update
            window = DQNWindow()
            for k in range(tc.total_env_steps // tc.n_envs):
                epsilon = linear_schedule(k * tc.n_envs, ac.eps_start, ac.eps_end, decay_steps)
                with timer.phase("act"):
                    actions = agent.act_epsilon(obs, epsilon, explore_rng)
                with timer.phase("env"):
                    next_obs, reward, terminated, truncated, info = env.step(actions)
                # True successor (not the post-reset observation) and the terminated
                # flag only: a starvation truncation is learned from as an ordinary
                # transition, so its target bootstraps.
                if buffer is not None:
                    buffer.add(obs, actions, reward, info["final_obs"], terminated)
                window.add_step(info, truncated)

                env_steps = (k + 1) * tc.n_envs
                if env_steps >= ac.learning_starts:
                    with timer.phase("update"):
                        if buffer is not None:
                            learn_from_buffer(k * tc.n_envs)
                        else:  # no replay: this vec-step's own transitions, once, one gradient step
                            own = Batch(obs=obs, actions=actions, rewards=reward, next_obs=info["final_obs"],
                                        terminated=terminated)
                            window.add_update(agent.update(own))
                obs = next_obs
                if env_steps % tc.eval_every == 0:
                    last_eval = evaluate_now(env_steps, epsilon, window)
                    window = DQNWindow()
    except FloatingPointError as err:
        # Mark the run as aborted (not merely interrupted) for the analysis;
        # the metrics file is already closed with the rows written so far.
        _write_json(run_dir / ABORTED_FILE, {
            "error": str(err),
            "env_steps": env_steps,
            "vec_steps": env_steps // tc.n_envs,
            "grad_steps": agent.grad_steps,
        })
        raise

    agent.save(run_dir / MODEL_FILE)  # with the optimiser state
    _save_final_eval(last_eval, run_dir)


@dataclass
class DQNWindow:
    """Training statistics of the DQN loop accumulated between two metrics rows."""

    episodes: int = 0
    score_sum: int = 0
    truncated: int = 0
    updates: int = 0
    stat_sums: dict[str, float] = field(default_factory=lambda: dict.fromkeys(UPDATE_STATS, 0.0))

    def add_step(self, info: Mapping[str, np.ndarray], truncated: np.ndarray) -> None:
        """One vec-step: the training episodes it finished, and how many of them by starvation."""
        done = info["done"]
        self.episodes += int(done.sum())
        self.score_sum += int(info["score"][done].sum())
        self.truncated += int(truncated.sum())

    def add_update(self, stats: Mapping[str, float]) -> None:
        """One gradient step: the statistics ``DQNAgent.update`` returned."""
        self.updates += 1
        for name in UPDATE_STATS:
            self.stat_sums[name] += stats[name]

    @property
    def mean_score(self) -> float | None:
        return self.score_sum / self.episodes if self.episodes else None

    def mean(self, name: str) -> float | None:
        """Mean of one update statistic over the window; ``None`` if there was no update."""
        return self.stat_sums[name] / self.updates if self.updates else None


class PhaseTimer:
    """Cumulative wall-clock seconds per phase of the DQN loop (the ``time_*``
    columns) and since its creation (``wall_time_s``).

    ``synchronize`` (``torch.cuda.synchronize`` on cuda, ``None`` on the CPU)
    waits for queued GPU work. It is called by :meth:`flush` only, at a
    metrics row, so the per-update cost is not inflated by a wait after every
    step; the time it takes is booked on the phase that queued the work.
    """

    def __init__(self, synchronize: Callable[[], None] | None = None) -> None:
        self.seconds = dict.fromkeys(PHASES, 0.0)
        self._synchronize = synchronize
        self._started = time.perf_counter()

    @contextmanager
    def phase(self, name: str) -> Iterator[None]:
        """Add the duration of the ``with`` block to phase ``name``."""
        t0 = time.perf_counter()
        try:
            yield
        finally:
            self.seconds[name] += time.perf_counter() - t0

    def flush(self, name: str) -> None:
        """Wait for queued GPU work (if any) and book the wait on phase ``name``."""
        if self._synchronize is not None:
            with self.phase(name):
                self._synchronize()

    @property
    def elapsed(self) -> float:
        return time.perf_counter() - self._started


def _evaluate_dqn(
    agent: "DQNAgent",
    buffer: ReplayBuffer | None,
    cfg: ExperimentConfig,
    metrics: "MetricsWriter",
    run_dir: Path,
    timer: PhaseTimer,
    env_steps: int,
    epsilon: float,
    window: DQNWindow,
    *,
    label: str,
    quiet: bool,
) -> EvalResult:
    """Greedy evaluation on the fixed evaluation boards; appends one metrics
    row and, with ``keep_checkpoints``, saves the evaluated weights."""
    ac, tc = cfg.agent, cfg.train
    timer.flush("update")  # the gradient steps since the previous row are complete before the clock is read
    with timer.phase("eval"):  # ends with the action values on the host, so nothing is left queued
        result = evaluate_batched(agent, cfg.env, tc.eval_episodes, tc.eval_seed, gamma=ac.gamma)
    if ac.keep_checkpoints:
        agent.save(run_dir / CHECKPOINT_DIR / f"step_{env_steps:08d}.pt", include_optimizer=False)
    row = {
        "env_steps": env_steps,
        "vec_steps": env_steps // tc.n_envs,
        "grad_steps": agent.grad_steps,
        "epsilon": epsilon,
        **eval_columns(result),
        **value_columns(result),
        "train_episodes": window.episodes,
        "train_mean_score": window.mean_score,
        "train_truncated": window.truncated,
        **{name: window.mean(name) for name in UPDATE_STATS},
        "buffer_size": len(buffer) if buffer is not None else 0,
        **{f"time_{phase}_s": round(timer.seconds[phase], 3) for phase in PHASES},
        "wall_time_s": round(timer.elapsed, 3),
    }
    metrics.write(row)
    if not quiet:
        print(_dqn_progress_line(label, tc.total_env_steps, row), flush=True)
    return result


def _dqn_run_info(cfg: ExperimentConfig) -> dict[str, Any]:
    """What a DQN run adds to the ``run`` block of ``config.yaml``: the torch
    version and the hardware the numbers were produced on."""
    import torch

    device = resolve_device(cfg.agent.device)  # the device DQNAgent will resolve to
    threads = cfg.agent.torch_threads if cfg.agent.torch_threads is not None else torch.get_num_threads()
    return {
        "torch": str(torch.__version__),  # a str subclass that YAML cannot represent
        "device": device.type,
        "cuda_device": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "torch_threads": threads,
    }


#: ``agent.kind`` -> trainer ``(cfg, seed, run_dir, quiet)``. A trainer writes
#: its metrics and checkpoints into ``run_dir`` and ``final_eval.json`` last.
_TRAINERS: dict[str, Callable[[ExperimentConfig, int, Path, bool], None]] = {
    "tabular_q": _train_tabular_q,
    "dqn": _train_dqn,
}
#: ``agent.kind`` -> extra keys of the ``run`` block of ``config.yaml``
#: (kinds without an entry add nothing).
_RUN_INFO: dict[str, Callable[[ExperimentConfig], dict[str, Any]]] = {
    "dqn": _dqn_run_info,
}


# ------------------------------------------------------------- entry point
def train(
    cfg: ExperimentConfig,
    seed: int,
    runs_dir: str | Path = "runs",
    argv: Sequence[str] | None = None,
    quiet: bool = False,
) -> Path:
    """Train one run and return its directory ``<runs_dir>/<exp>/<seed>/``.

    ``argv`` is recorded in ``config.yaml`` (the command-line arguments when
    called from :func:`main`). ``ValueError`` for an invalid config or seed or
    an agent kind without a trainer; ``FileExistsError`` if the run exists;
    ``FloatingPointError`` if a DQN update produced a non-finite loss (the
    run directory is left without ``final_eval.json``).
    """
    cfg = config_from_dict(config_to_dict(cfg))  # a hand-built config gets the checks of a file
    trainer = _TRAINERS.get(cfg.agent_kind)
    if trainer is None:
        raise ValueError(f"no trainer for agent.kind {cfg.agent_kind!r} (known: {', '.join(_TRAINERS)})")
    env_seed = env_seed_for(seed, cfg.train.n_envs)
    run_dir = Path(runs_dir) / cfg.exp / str(seed)
    _start_run(cfg, seed, env_seed, run_dir, argv)
    if not quiet:
        print(f"[{cfg.exp}/{seed}] {cfg.agent_kind} -> {run_dir}  (env_seed {env_seed})", flush=True)
    trainer(cfg, seed, run_dir, quiet)
    if not quiet:
        print(f"[{cfg.exp}/{seed}] finished -> {run_dir}", flush=True)
    return run_dir


# ------------------------------------------------------ run-directory files
def _start_run(cfg: ExperimentConfig, seed: int, env_seed: int, run_dir: Path, argv: Sequence[str] | None) -> None:
    """Claim the run directory by writing ``config.yaml``; refuse an existing run."""
    existing = [name for name in RUN_FILES if (run_dir / name).exists()]
    if existing:
        raise FileExistsError(
            f"{run_dir} already holds a run ({', '.join(existing)}); runs are never overwritten - "
            f"use another --exp or --seed, or, to repeat an interrupted run under the same name, move that "
            f"directory out of its experiment directory first (for example to runs/_interrupted/)"
        )
    record = config_to_dict(cfg)
    record["run"] = {
        "seed": seed,
        "env_seed": env_seed,
        "git": git_hash(),
        "started": datetime.now().astimezone().isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "platform": platform.platform(),
        "argv": None if argv is None else [str(arg) for arg in argv],
        **_RUN_INFO.get(cfg.agent_kind, lambda _: {})(cfg),
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    # Mode "x" fails if the file exists: two processes cannot both claim the run.
    with open(run_dir / CONFIG_FILE, "x", encoding="utf-8") as f:
        f.write("# Written by snake_rl.train before training started. To repeat this run:\n")
        f.write("#   python -m snake_rl.train --config <this file> --seed <run.seed> --exp <new name>\n")
        yaml.safe_dump(record, f, sort_keys=False)


def _write_json(path: Path, payload: Mapping[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(dict(payload), f, indent=2)
        f.write("\n")


def _save_final_eval(result: EvalResult, run_dir: Path) -> None:
    """``final_eval.json`` marks the run as finished (the analysis and the
    batch runner look for it), so it must never exist half-written: write
    next to it, then rename."""
    partial_file = run_dir / (FINAL_EVAL_FILE + ".tmp")
    result.save(partial_file)
    os.replace(partial_file, run_dir / FINAL_EVAL_FILE)


class MetricsWriter:
    """Append-only CSV with a fixed header. The header is written on creation
    and every row is flushed, so an interrupted run keeps its curve up to the
    last evaluation. Refuses to overwrite an existing file."""

    def __init__(self, path: str | Path, columns: Sequence[str]) -> None:
        self.columns = tuple(columns)
        self._file = open(path, "x", newline="", encoding="utf-8")
        self._writer = csv.writer(self._file, lineterminator="\n")
        self._writer.writerow(self.columns)
        self._file.flush()

    def write(self, row: Mapping[str, Any]) -> None:
        """Append one row; ``row`` must have exactly the columns of the header."""
        if set(row) != set(self.columns):
            raise KeyError(f"metrics row keys {sorted(row)} do not match the header {sorted(self.columns)}")
        self._writer.writerow([_cell(row[name]) for name in self.columns])
        self._file.flush()

    def close(self) -> None:
        self._file.close()

    def __enter__(self) -> "MetricsWriter":
        return self

    def __exit__(self, *exc_info: object) -> None:
        self.close()


def _cell(value: Any) -> str:
    """CSV text of one value: empty for ``None``, ``repr`` for floats (exact
    round trip), plain digits for integers."""
    if value is None:
        return ""
    if isinstance(value, (bool, np.bool_)):
        raise TypeError("booleans are not metrics values")
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    return repr(float(value))


def eval_columns(result: EvalResult) -> dict[str, Any]:
    """The ``eval_*`` metrics columns of one evaluation (``EvalResult.summary()``)."""
    s = result.summary()
    frac = s["cause_fractions"]
    return {
        "eval_mean_score": s["mean_score"],
        "eval_ci95": s["ci95"],
        "eval_median_score": s["median_score"],
        "eval_max_score": s["max_score"],
        "eval_mean_steps": s["mean_steps"],
        "eval_frac_wall": frac["wall"],
        "eval_frac_self": frac["self"],
        "eval_frac_starve": frac["starve"],
        "eval_frac_win": frac["win"],
    }


def value_columns(result: EvalResult) -> dict[str, float]:
    """The ``eval_value_* / eval_return_*`` columns (``EvalResult.value_summary()``)."""
    values = result.value_summary()
    if values is None:
        raise ValueError("the evaluation carries no value estimates: evaluate with gamma")
    return {f"eval_{name}": values[name] for name in ("value_start", "return_start", "value_visited", "return_visited")}


def _progress_line(label: str, total_env_steps: int, row: Mapping[str, Any]) -> str:
    def optional(value: float | None, spec: str) -> str:
        return "-" if value is None else format(value, spec)

    width = len(f"{total_env_steps:,}")
    return (
        f"[{label}] {row['env_steps']:>{width},}/{total_env_steps:,} steps  eps {row['epsilon']:.3f}  "
        f"eval {row['eval_mean_score']:.2f} +- {row['eval_ci95']:.2f}  "
        f"wall {row['eval_frac_wall']:.2f} self {row['eval_frac_self']:.2f} "
        f"starve {row['eval_frac_starve']:.2f} win {row['eval_frac_win']:.2f}  "
        f"train {optional(row['train_mean_score'], '.2f')} ({row['train_episodes']} ep)  "
        f"|td| {optional(row['mean_abs_td'], '.4f')}  {row['wall_time_s']:.1f}s"
    )


def _dqn_progress_line(label: str, total_env_steps: int, row: Mapping[str, Any]) -> str:
    def optional(value: float | None, spec: str) -> str:
        return "-" if value is None else format(value, spec)

    width = len(f"{total_env_steps:,}")
    return (
        f"[{label}] {row['env_steps']:>{width},}/{total_env_steps:,} steps  eps {row['epsilon']:.3f}  "
        f"eval {row['eval_mean_score']:.2f} +- {row['eval_ci95']:.2f}  "
        f"wall {row['eval_frac_wall']:.2f} self {row['eval_frac_self']:.2f} "
        f"starve {row['eval_frac_starve']:.2f} win {row['eval_frac_win']:.2f}  "
        f"train {optional(row['train_mean_score'], '.2f')} ({row['train_episodes']} ep)  "
        f"loss {optional(row['loss'], '.4f')}  q {optional(row['mean_q'], '.3f')}  "
        f"{row['grad_steps']:,} updates  {row['wall_time_s']:.1f}s"
    )

### 5. Training in This Notebook

The code above trains here. `TRAIN_MODE = "demo"` runs the 8 x 8 tabular configuration of the report (500,000
environment steps, about ten seconds) and the two short DQN configurations of the repository's smoke tests (16,000
steps on observation A with 64-unit layers; 48,000 steps on observation B with a 16 / 32-channel CNN), about two
minutes on a CPU in all. `TRAIN_MODE = "full"` runs one seed of the report's 8 x 8 DQN configurations instead
(512,000 steps, 125,504 gradient steps each; about 5 minutes for the MLP and 40 minutes for the CNN on the training
machine's CPU). The runs are written under `runs_notebook/`, apart from the logs of the study; a finished run is
reused when the notebook is run again.

#### 5.1 Settings

In [ ]:
# "demo" = the quick configurations (about two minutes on a CPU); "full" = one seed of the 8 x 8 configurations of the report
TRAIN_MODE = "demo"
NOTEBOOK_RUNS = "runs_notebook"          # the runs of this notebook, apart from the logs of the study under runs/
GPU = ("agent.device=cuda",) if torch.cuda.is_available() else ()

CONFIGS = {
    "demo": {"tabular": ("configs/tabular_q_8x8.yaml", ()),
             "dqn_A": ("configs/smoke_8x8.yaml", ()),
             "dqn_B": ("configs/smoke_B_8x8.yaml", ())},
    "full": {"tabular": ("configs/tabular_q_8x8.yaml", ()),
             "dqn_A": ("configs/dqn_A_8x8.yaml", GPU),
             "dqn_B": ("configs/dqn_B_8x8.yaml", GPU)},
}[TRAIN_MODE]


def train_or_reuse(config_file, overrides=(), seed=0):
    # one run of config_file under NOTEBOOK_RUNS: a finished run is reused, an unfinished one is removed and repeated
    exp_cfg = load_config(config_file, overrides)
    run_dir = os.path.join(NOTEBOOK_RUNS, exp_cfg.exp, str(seed))
    if os.path.exists(os.path.join(run_dir, FINAL_EVAL_FILE)):
        print(f"[{exp_cfg.exp}/{seed}] finished run found -> {run_dir}")
        return run_dir
    shutil.rmtree(run_dir, ignore_errors=True)
    return str(train(exp_cfg, seed, NOTEBOOK_RUNS))


print(f"TRAIN_MODE = {TRAIN_MODE}:", ", ".join(f"{k} <- {v[0]}" for k, v in CONFIGS.items()))

#### 5.2 The reference players on the evaluation boards

In [ ]:
# Random, BFS-greedy and Hamiltonian players on the 100 evaluation boards of the 8 x 8 study (episode i plays the board of seed 10000 + i)
board_cfg = load_config(CONFIGS["tabular"][0]).env
baseline_results = [evaluate(make_policy(name), board_cfg, n_episodes=100, seed=DEFAULT_EVAL_SEED) for name in BASELINE_NAMES]
os.makedirs(os.path.join(NOTEBOOK_RUNS, "baselines"), exist_ok=True)
baseline_files = [str(r.save(os.path.join(NOTEBOOK_RUNS, "baselines", f"{r.policy}_W8.json"))) for r in baseline_results]
display(Markdown(markdown_table(baseline_results)))

#### 5.3 Tabular Q-learning

In [ ]:
# Tabular Q-learning on observation A, 8 x 8: the configuration of the report (500,000 environment steps on 50 boards, one evaluation every 25,000 steps)
run_dirs = {}
run_dirs["tabular"] = train_or_reuse(*CONFIGS["tabular"])

#### 5.4 DQN on observation A

In [ ]:
# DQN with the MLP on the 11 features: the smoke configuration in demo mode, the report's 8 x 8 configuration in full mode
run_dirs["dqn_A"] = train_or_reuse(*CONFIGS["dqn_A"])

#### 5.5 DQN on observation B

In [ ]:
# DQN with the CNN on the 8-plane grid: the smoke configuration in demo mode, the report's 8 x 8 configuration in full mode
run_dirs["dqn_B"] = train_or_reuse(*CONFIGS["dqn_B"])

#### 5.6 Learning curves and the final evaluation

In [ ]:
# Evaluation curves of the three runs against the reference players, then the final greedy evaluation of every player on the same boards
labels = {"tabular": "tabular Q-learning (A)", "dqn_A": "DQN (A, MLP)", "dqn_B": "DQN (B8, CNN)"}
baselines = [(r.policy, f) for r, f in zip(baseline_results, baseline_files)]
for keys, png in ((("tabular",), "curve_tabular.png"), (("dqn_A", "dqn_B"), "curve_dqn.png")):
    out = plot_learning_curves([(labels[k], os.path.dirname(run_dirs[k])) for k in keys], baselines,
                               out=os.path.join(NOTEBOOK_RUNS, "figures", png), ylabel="mean score (greedy evaluation)")
    display(Image(filename=str(out)))

final_results = [EvalResult.load(os.path.join(d, FINAL_EVAL_FILE)) for d in run_dirs.values()]
for r, k in zip(final_results, run_dirs):
    r.policy = labels[k]
display(Markdown(markdown_table(final_results + baseline_results)))

#### 5.7 One greedy episode of the trained DQN

In [ ]:
# The DQN trained on observation A plays the first evaluation board greedily; the board every ten steps and at the end (H head, o body, * food)
agent_A = DQNAgent.load(os.path.join(run_dirs["dqn_A"], MODEL_FILE), device="cpu")
board = SnakeEnv(load_config(CONFIGS["dqn_A"][0]).env, render_mode="ansi")
obs, info = board.reset(seed=DEFAULT_EVAL_SEED)
frames, done = [], False
while not done:
    if board.steps % 10 == 0:
        frames.append(f"step {board.steps}, score {board.score}\n{board.render()}")
    obs, reward, terminated, truncated, info = board.step(agent_A.act(obs))
    done = terminated or truncated
frames.append(f"step {board.steps}, score {board.score}, end: {info['cause']}\n{board.render()}")
print("\n\n".join(frames[:4] + frames[-1:]))

### 6. Training Logs of the Full Study

In [ ]:
# Training logs: the local runs/ directory, or runs_logs.zip (scripts/export_logs.py) from LOGS_URL or the Colab upload dialog
LOGS_URL = ""               # link to runs_logs.zip; empty: upload it on Colab, or use the local runs/
LOGS_ZIP = "runs_logs.zip"


def experiment_dirs():
    return sorted(d for d in glob.glob("runs/*/") if glob.glob(os.path.join(d, "*", "final_eval.json")))


if not experiment_dirs():
    if LOGS_URL:
        urllib.request.urlretrieve(LOGS_URL, LOGS_ZIP)
    elif IN_COLAB and not os.path.exists(LOGS_ZIP):
        from google.colab import files

        LOGS_ZIP = next(iter(files.upload()))
    elif not os.path.exists(LOGS_ZIP):
        raise FileNotFoundError("no runs/ and no runs_logs.zip: set LOGS_URL or copy the zip next to this notebook")
    with zipfile.ZipFile(LOGS_ZIP) as zf:
        zf.extractall(".")  # entries are runs/<experiment>/<seed>/<file>

print(len(experiment_dirs()), "experiment directories with finished runs under runs/")

### 7. Experiment Matrix

The 64 configurations of the study are defined in `snake_rl/analysis/matrix.py`; the status table lists the finished
seeds of every row and checks each run's recorded `config.yaml` against the row that launched it.

In [ ]:
# Finished seeds per configuration, and every run's config.yaml checked against the matrix
!{sys.executable} -m snake_rl.analysis.matrix --status

### 8. Building the Report Outputs

#### 8.1 Generated tables and figures

In [ ]:
# Every generated table and figure (runs/report/); items whose experiments are missing are skipped and listed in runs/report/index.md
!{sys.executable} -m snake_rl.analysis.report --runs-dir runs --out-dir runs/report

#### 8.2 Composite figures

In [ ]:
# Figures 2, 9, 11 and 12 of the report: score distributions and death causes from the run directories, heat-maps from the saved behaviour analyses
sys.path.insert(0, "scripts")
from report_extra_figures import PANELS

needed = sorted({name for _, _, panels in PANELS for _, name in panels})
missing = [name for name in needed if not os.path.isdir(os.path.join("runs/behaviour", name))]
if missing:
    print("behaviour analyses missing under runs/behaviour:", ", ".join(missing))
    print("export the logs with --include-behaviour, or run snake_rl.analysis.behaviour for these players first")
else:
    !{sys.executable} scripts/report_extra_figures.py --out-dir runs/report/figures

#### 8.3 The statistics of the report's tables

In [ ]:
# Tables 3 to 9 and C1 of the report from the run directories; a table is skipped when its experiments are missing
tables, skipped = rt.build("runs")
for name in rt.TABLES:
    print(f"{name}: {'computed' if name in tables else 'skipped (' + skipped[name] + ')'}")

#### 8.4 The interface screenshot

In [ ]:
# Figure 1 is a screenshot of the GUI (scripts/gui_screenshots.py); rendered here when the two runs it plays are present with their weights
FIGURE1 = "runs/figures/partD_gui_agent.png"
if os.path.exists(FIGURE1):
    print("present:", FIGURE1)
elif all(os.path.exists(p) for p in ("runs/dqn_A_W10/0/model.pt", "runs/tabular_q_W10/0/q_table.npz")):
    !{sys.executable} scripts/gui_screenshots.py --out-dir runs/figures
else:
    print("not rendered: the screenshot needs the weights of runs/dqn_A_W10/0 and runs/tabular_q_W10/0")

### 9. The Report, Table by Table and Figure by Figure

Each item below is one table or figure of the report, with the report's caption. Tables 1 and 2 are text (the
training settings and the summary of the answers) and are shown as the report source has them; every other table is
computed from the logs in Section 8.3, and the figures are the files that Section 8 wrote.

In [ ]:
# Captions, figure files and source tables of the report, and the two display helpers
captions = rt.report_captions(REPORT_MD)
figure_files = rt.report_figures(REPORT_MD)
source_tables = rt.report_tables(REPORT_MD)


def show_table(name, markdown=None):
    # a computed table (markdown) or, without one, the table of the report source; with the report's caption
    if markdown is None:
        if name not in tables and name in skipped:
            print(f"{name}: not computed ({skipped[name]})")
            return
        rows = source_tables[name]
        markdown = rt.markdown_table(rows[0], rows[1:])
    display(Markdown(f"**{name}.** {captions[name]}\n\n{markdown}"))


def show_figure(name, files=None):
    # the file(s) of a figure of the report with the report's caption
    files = figure_files[name] if files is None else files
    for f in files:
        if os.path.exists(f):
            display(Image(filename=f))
        else:
            print(f"{name}: missing {f}")
    display(Markdown(f"**{name}.** {captions[name]}"))

#### 9.1 The interface (Section 3.6): Figure 1

In [ ]:
show_figure("Figure 1")

#### 9.2 Training settings (Section 4): Table 1

In [ ]:
show_table("Table 1")

#### 9.3 The questions and the answers (Section 6): Table 2

In [ ]:
show_table("Table 2")

#### 9.4 The scale and the tabular reference (Section 6.1): Table 3 and Figure 2

In [ ]:
show_table("Table 3", tables.get("Table 3"))
show_figure("Figure 2")

#### 9.5 State representation (Section 6.2): Table 4 and Figure 3

In [ ]:
show_table("Table 4", tables.get("Table 4"))
show_figure("Figure 3")

#### 9.6 Discount factor (Section 6.3): Table 5 and Figure 4

In [ ]:
show_table("Table 5", tables.get("Table 5"))
show_figure("Figure 4")

#### 9.7 The stabilisers (Section 6.4): Table 6, Figures 5 and 6

In [ ]:
show_table("Table 6", tables.get("Table 6"))
show_figure("Figure 5")
show_figure("Figure 6")

#### 9.8 Double DQN, Dueling and prioritised replay (Section 6.5): Table 7 and Figure 7

In [ ]:
show_table("Table 7", tables.get("Table 7"))
show_figure("Figure 7")

#### 9.9 Reward design (Section 6.6): Table 8, Figures 8 and 9

In [ ]:
show_table("Table 8", tables.get("Table 8"))
show_figure("Figure 8")
show_figure("Figure 9")

#### 9.10 Reward hacking, the probe (Section 6.7): Figure 10

In [ ]:
show_figure("Figure 10")

#### 9.11 Behaviour (Section 6.8): Table 9, Figures 11 to 13

In [ ]:
show_table("Table 9", tables.get("Table 9"))
show_figure("Figure 11")
show_figure("Figure 12")
show_figure("Figure 13")

#### 9.12 Every configuration (Appendix B): Tables B1 and B2

In [ ]:
# The columns of the report from the generated summary tables of both boards
for name, W in (("Table B1", 8), ("Table B2", 10)):
    show_table(name, rt.appendix_table(f"runs/report/tables/summary_W{W}.md"))

#### 9.13 The tabular sweep (Appendix C): Table C1

In [ ]:
show_table("Table C1", tables.get("Table C1"))

### 10. Check Against the Report

Every computed table is compared with the table in the report source cell by cell, and every figure file of the
report must exist.

In [ ]:
# Computed tables against the report source, and the figure files of the report
problems = 0
for name in rt.TABLES:
    if name not in tables:
        print(f"{name}: not computed ({skipped[name]})")
        problems += 1
        continue
    diffs = rt.compare_with_report(name, tables[name], REPORT_MD)
    problems += len(diffs)
    print(f"{name}: {'identical to the report' if not diffs else str(len(diffs)) + ' differences'}")
    for d in diffs:
        print("  " + d)
for name, files in figure_files.items():
    missing_files = [f for f in files if not os.path.exists(f)]
    if missing_files:
        problems += 1
        print(f"{name}: missing {', '.join(missing_files)}")
    else:
        print(f"{name}: {', '.join(files)}")
print("RESULT:", "every table and figure of the report is reproduced" if problems == 0 else f"{problems} problem(s)")

### 11. Behaviour Analysis of One Run

`snake_rl.analysis.behaviour` plays 100 evaluation episodes with a trained network and writes the visit heat-map, the
loop / shaping statistics per episode and GUI-replayable recordings of the longest loops
(`python -m snake_rl.gui.app --replay <file>`). The saved analyses behind Table 9 and Figures 11 to 13 were made this
way. The run analysed is the gamma = 0.99 DQN of the report when its weights are present (the logs zip carries none),
otherwise the DQN trained on observation B in Section 5.

In [ ]:
# Behaviour analysis of one run: 100 greedy episodes, heat-map, loop statistics and recordings of the longest loops
candidates = ("runs/dqn_B8_W8_gamma099/0", run_dirs["dqn_B"])
RUN_DIR = next(d for d in candidates if os.path.exists(os.path.join(d, MODEL_FILE)))
OUT_DIR = os.path.join(NOTEBOOK_RUNS, "behaviour", os.path.basename(os.path.dirname(RUN_DIR)))

!{sys.executable} -m snake_rl.analysis.behaviour --player {RUN_DIR} --checkpoint final --device {DEVICE} --out-dir {OUT_DIR}
display(Image(filename=os.path.join(OUT_DIR, "heatmap.png")))
display(Image(filename=os.path.join(OUT_DIR, "last_steps.png")))
with open(os.path.join(OUT_DIR, "summary.json"), encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2)[:2000])